1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path.home() / 'ai_models'

'''
Select the GPU IDs available to this notebook.
指定這份 Notebook 可使用的 GPU ID。
'''
GPU_IDS = [0]


'''
Select the GPUs for the generation model.
指定生成模型使用的 GPU ID。
'''
MODEL_GPU_IDS = [0]


'''
Select the GPU for text encoders, VAE, and codec models.
指定文字編碼器、VAE 與 Codec 等輔助模型使用的 GPU ID。
'''
AUX_GPU_ID = 0


'''
Derive the primary model GPU for existing loaders; no manual edit needed.
依上述設定取得 Loader 使用的主要生成 GPU，不需要手動修改。
'''
MODEL_GPU_ID = MODEL_GPU_IDS[0]


'''
Count the selected GPUs for the existing initialization flow.
計算指定的 GPU 數量，供原有初始化流程使用。
'''
GPU_COUNT = len(GPU_IDS)

'''
Reject duplicate, out-of-range, or inconsistent GPU assignments.
拒絕重複、超出設定範圍或互相衝突的 GPU 編號。
'''
assert GPU_IDS and len(set(GPU_IDS)) == len(GPU_IDS) and all(type(i) is int and i >= 0 for i in GPU_IDS)
assert MODEL_GPU_IDS and len(set(MODEL_GPU_IDS)) == len(MODEL_GPU_IDS)
assert set(MODEL_GPU_IDS).issubset(GPU_IDS) and AUX_GPU_ID in GPU_IDS
ROOT.mkdir(parents=True, exist_ok=True)

'''
Validate the selected IDs against the ROCm/CUDA-visible PyTorch devices.
比對 ROCm／CUDA 版 PyTorch 可見的 GPU 編號，避免錯誤分配。
'''
def validate_torch_gpus(sharding=False):
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('找不到 PyTorch GPU。請先在 Ubuntu 安裝符合 GPU 型號的 ROCm PyTorch。')
    available = torch.cuda.device_count()
    if max(GPU_IDS) >= available:
        raise RuntimeError(f'GPU_IDS={GPU_IDS}，但 PyTorch 只看得到 {available} 張 GPU。')
    if len(MODEL_GPU_IDS) > 1 and not sharding:
        raise RuntimeError('這套 Loader 尚未驗證跨 GPU 主模型切分；MODEL_GPU_IDS 請先只指定一張。')
    backend = 'AMD ROCm' if torch.version.hip else 'NVIDIA CUDA'
    print(f'{backend}：選擇 {GPU_IDS}；生成 {MODEL_GPU_IDS}；輔助 cuda:{AUX_GPU_ID}')
    for gpu_id in GPU_IDS:
        memory = torch.cuda.get_device_properties(gpu_id).total_memory / 1024**3
        print(f'GPU {gpu_id}: {torch.cuda.get_device_name(gpu_id)} ({memory:.1f} GiB)')


        '''
        Warn if the selected device has less memory than the 32GB profile expects.
        若選用的 GPU 容量低於 32GB 配置需求，提醒可能需要更積極的 Offload。
        '''
        if memory < 27:
            '''
            Explain the available-VRAM limitation without blocking execution.
            提醒可用 VRAM 較少，但不阻擋使用者繼續執行。
            '''
            print('提醒：此 Notebook 預設供約 32GB VRAM 的 GPU 使用；目前容量可能需要更積極的 Offload。')


COMFY = ROOT / 'ComfyUI-MiniMaxH3'

'''
Store large downloaded GGUF weights in scratch storage.
設定 GGUF 模型權重的暫存磁碟位置，減少工作空間占用。
'''
STORE = ROOT / 'minimax_h3_models'
PORT = 8193  # 本機可用 Port
H3_COMFY_REF = 'v0.45.0'

'''
Set the Hugging Face cache directory before any model downloads.
在下載模型之前指定 Hugging Face 快取目錄，避免大型檔案占滿工作空間。
'''
HF_CACHE_DIR = ROOT / 'minimax_h3_hf'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the shared GGUF quantization. Larger formats may require CPU offload on smaller VRAM cards.
選擇共用 GGUF 量化精度；越大的格式越可能需要 CPU Offload。
'''
QUANT = 'Q5_0'  # 'Q4_K', 'Q5_0', 'Q6_K', 'Q8_0'
DEFAULT_QUANT = QUANT

'''
Set the primary model GPU memory budget; leave headroom for runtime tensors.
設定生成 GPU 的權重預算（GiB），保留中間張量與執行時的空間。
'''
GPU0_MODEL_GIB = 24.0  # 單張 32GB 卡的主要 VRAM 預算（GiB），其餘供暫存／Offload
GPU1_MODEL_GIB = 24.0  # 輔助模型監控參考值；同一張卡不會額外增加 24 GiB

'''
Select the text-encoder weight placement strategy.
設定文字編碼器權重管理：dynamic 分批調度較節省 GPU 記憶體；streaming 使用時整體載入；resident 盡量常駐 GPU 記憶體。
'''
TEXT_MODE = 'dynamic'  # 'dynamic', 'streaming', 'resident'
TEXT_GPU = f'cuda:{AUX_GPU_ID}'  # 輔助 GPU，可與生成 GPU 相同
NODE_CACHE_MODE = 'classic'  # 'classic', 'none'

'''
Select the optional remote FIFO API port; change only if the port conflicts.
指定選用 FIFO 遠端 API 的本機連接埠；只有發生 Port 衝突時才需要修改。
'''
REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_MiniMax_H3_影片生成_Ubuntu'
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID

'''
Limit how many remote jobs wait for the single GPU worker.
設定 FIFO 最多等待的工作數量；超出時回傳 HTTP 429，client 會延遲重試。
'''
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2

'''
Mark exceptionally long remote inference as stalled, without forcibly starting a second GPU job.
設定遠端推論超時警戒值；卡住時不強行啟動第二筆 GPU 推論。
'''
REMOTE_STALL_HOURS = 2
GGUF_REPO = 'robertbschuette/MiniMax-H3-GGUF'
R2V_REF_DIR = ROOT / 'minimax_h3_r2v_references'

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= PORT <= 65535 and PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')


2. 安裝環境與匯入套件


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
validate_torch_gpus()
for i in GPU_IDS:
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Define the repositories needed for the local workflow.
定義本機 Workflow 需要的程式庫。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/leejet/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
}
old_commits = {}
for directory, url in repositories.items():
    if directory.exists() and not (directory / '.git').is_dir():
        raise RuntimeError(f'目錄不是 Git 倉庫：{directory}')
    if (directory / '.git').is_dir():
        old_commits[directory] = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        command = ['git', 'clone', '--depth', '1']

        '''
        Choose the release tag only for ComfyUI core.
        只有 ComfyUI 核心指定發行版標籤。
        '''
        if directory == COMFY:
            command += ['--branch', H3_COMFY_REF]
        subprocess.run(command + [url, str(directory)], check=True)

    '''
    The archived or patched GGUF loader may contain local compatibility changes.
    GGUF 載入器可能含有必要的本機相容性修正。
    '''
    if directory.name == 'ComfyUI-GGUF':
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()
        if not dirty:
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
        else:
            print('GGUF 載入器已有本機修改，本次略過自動更新。')

'''
Retrieve the official ComfyUI release on repeat runs.
重跑時同樣取得官方 ComfyUI 發行版本。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', H3_COMFY_REF], check=True)
subprocess.run(['git', '-C', str(COMFY), 'checkout', '--quiet', '--detach', H3_COMFY_REF], check=True)
H3_CORE_UPDATED = any(
    directory in old_commits and old_commits[directory] != subprocess.check_output(
        ['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True,
    ).strip()
    for directory in (COMFY, COMFY / 'custom_nodes/ComfyUI-GGUF')
)

'''
Update ClipProj to its current compatible implementation if this folder already existed.
若 ClipProj 資料夾原本就存在，同步更新到支援新版投影矩陣的程式。
'''
clipproj_dir = COMFY / 'custom_nodes/ComfyUI-ClipProj'
clipproj_before = subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(clipproj_dir), 'fetch', '--quiet', '--depth', '1', 'origin', 'main'], check=True)

'''
Switch the local custom node to the retrieved revision for consistent code and weights.
切換到取得的版本，確保節點程式與投影矩陣相容。
'''
subprocess.run(['git', '-C', str(clipproj_dir), 'checkout', '--quiet', '--force', 'FETCH_HEAD'], check=True)
H3_CLIPPROJ_UPDATED = clipproj_before != subprocess.check_output(['git', '-C', str(clipproj_dir), 'rev-parse', 'HEAD'], text=True).strip()
clipproj_source = (clipproj_dir / 'clipproj_nodes.py').read_text(encoding='utf-8')

'''
Fail before loading models if an outdated fork lacks the required interfaces.
若外掛版本過舊且缺少必要介面，在載入模型前停止。
'''
assert 'def clip_types(' in clipproj_source and 'build_residual' in clipproj_source, 'ClipProj 缺少新版必要功能，請檢查安裝版本。'
constraint = ROOT / 'minimax_h3_torch_constraints.txt'
constraint.write_text(f'torch=={torch.__version__}\ntransformers>=4.50.3,<5.0\n', encoding='utf-8')

'''
Install ComfyUI dependencies and the GGUF backend.
安裝 ComfyUI 相依套件與 GGUF 後端。
'''
for requirements in [COMFY / 'requirements.txt', COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt']:
    if requirements.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirements)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), 'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)
print('ComfyUI 與 MiniMax H3 節點安裝完成。')

'''
Import a small regex helper for the H3 GGUF architecture registration.
載入簡單的正規表示式工具，以註冊 H3 GGUF 架構。
'''
import re
gguf_loader_path = COMFY / 'custom_nodes/ComfyUI-GGUF/loader.py'
assert gguf_loader_path.is_file(), f'找不到 GGUF 載入器：{gguf_loader_path}'

'''
Read the exact installed GGUF implementation instead of assuming a fixed version.
讀取實際安裝的 GGUF 原始碼，不假設版本永遠固定。
'''
gguf_source = gguf_loader_path.read_text(encoding='utf-8')
arch_match = re.search(r'IMG_ARCH_LIST\s*=\s*\{[^}]*\}', gguf_source)
assert arch_match, '無法定位 IMG_ARCH_LIST；請檢查 ComfyUI-GGUF 版本。'

'''
Record whether the running ComfyUI process would require a reload.
記錄是否需要讓已啟動的 ComfyUI 重新載入節點。
'''
H3_RESTART_REQUIRED = H3_CLIPPROJ_UPDATED or H3_CORE_UPDATED
if '"minimax_h3"' not in arch_match.group(0):
    previous_arch_list = arch_match.group(0)[:-1].rstrip()
    separator = '' if previous_arch_list.endswith(('{', ',')) else ','
    updated_arch_list = f'{previous_arch_list}{separator} "minimax_h3"}}'
    gguf_source = gguf_source[:arch_match.start()] + updated_arch_list + gguf_source[arch_match.end():]
    gguf_loader_path.write_text(gguf_source, encoding='utf-8')
    H3_RESTART_REQUIRED = True
    print('已加入 MiniMax-H3 GGUF 架構相容設定。')

print('✓ 第 2 節完成：ComfyUI、GGUF、ClipProj 與 WebSocket 工具已安裝／檢查。')


3. 下載量化模型（Q4／Q5／Q6／Q8，權重保留於 `/tmp`）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import shutil
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')

'''
Disable extra Xet chunk caching to conserve temporary disk space.
停用額外的 Xet 分塊快取，節省暫存空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
from huggingface_hub import hf_hub_download

'''
Choose the generation mode from the four supported task strings. flf2v = first-and-last-frame-to-video; r2v = multi-reference-to-video (Ref2VA model). Available choices: t2v, i2v, flf2v, r2v.
從四種合法字串中選擇生成模式。 t2v = text-to-video (文字生成影片); i2v = image-to-video (首張圖片生成影片). t2v 為文生影片；i2v 為首張圖片生成影片。 flf2v 為首尾影格生成影片；r2v 為多張參考圖生成影片（需使用 Ref2VA 權重）。 可選值：t2v、i2v、flf2v、r2v。
'''
TASK = globals().get('TASK', 't2v')  # 't2v', 'i2v', 'l2v', 'flf2v', 'r2v'
assert TASK in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'
MODEL_VARIANT = 'ref2va_pruned' if TASK == 'r2v' else 'fl2va_pruned'

'''
Use edited task parameters when the corresponding settings cell has run.
若已執行對應的參數 Cell，則優先採用使用者修改後的量化選項。
'''
task_settings = globals().get('TASK_CONFIGS', {}).get(TASK, {})
FL2VA_QUANT_SIZES_GIB = {'Q4_K': 10.64, 'Q5_0': 12.97, 'Q6_K': 15.45, 'Q8_0': 19.97}
REF2VA_QUANT_SIZES_GIB = {'Q4_K': 10.60, 'Q5_0': 12.94, 'Q6_K': 15.42, 'Q8_0': 19.94}

'''
Select the size table matching the actual diffusion model family.
依照真正要使用的模型種類選擇大小對照表。
'''
QUANT_SIZES_GIB = REF2VA_QUANT_SIZES_GIB if TASK == 'r2v' else FL2VA_QUANT_SIZES_GIB
assert QUANT in QUANT_SIZES_GIB, f'不支援 QUANT={QUANT!r}；請選擇 {list(QUANT_SIZES_GIB)}。'

'''
Build the actual published filename, never reusing FL2VA weights for R2V.
根據實際發布檔名選擇模型，R2V 絕不沿用 FL2VA 權重。
'''
DIT_NAME = f'minimax_h3_{MODEL_VARIANT}-{QUANT}.gguf'
models = [
    (GGUF_REPO, DIT_NAME, 'diffusion_models'),
    ('Comfy-Org/Krea-2', 'text_encoders/qwen3vl_4b_fp8_scaled.safetensors', 'text_encoders'),
    ('NicoLab28/ClipProj-MiniMax-H3', 'mmh3-4b-ClipProj-v3.1-mlp.safetensors', 'clip_projections'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
AUX_SIZES_GIB = {
    'qwen3vl_4b_fp8_scaled.safetensors': 4.88,
    'mmh3-4b-ClipProj-v3.1-mlp.safetensors': 0.47,
    'minimax_h3_video_vae_fp16.safetensors': 4.85,
    'minimax_h3_audio_vae_fp32.safetensors': 0.56,
}

'''
Keep the valid downloaded GGUF from either the current or historical folder.
可沿用目前或舊版資料夾中已完整下載的 GGUF 權重。
'''
GGUF_CACHE_PATHS = (STORE / 'diffusion_models' / DIT_NAME, STORE / 'unet_gguf' / DIT_NAME)
missing_gib = 0.0
for repo, filename, folder in models:
    size_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
    candidates = GGUF_CACHE_PATHS if folder == 'diffusion_models' else (STORE / folder / Path(filename).name,)
    complete = any(path.is_file() and 0.97 * size_gib <= path.stat().st_size / 1024**3 <= 1.03 * size_gib for path in candidates)

    '''
    Count only components that still need to be downloaded.
    只累計尚未完整下載的模型大小。
    '''
    if not complete:
        missing_gib += size_gib

'''
Read the free space of the scratch filesystem used for all model files.
讀取實際存放所有模型權重的暫存磁碟可用容量。
'''
tmp_free_gib = shutil.disk_usage(ROOT).free / 1024**3
assert tmp_free_gib >= missing_gib + 3, (
    f'模型儲存空間不足：仍需下載約 {missing_gib:.2f} GiB，'
    f'目前剩餘 {tmp_free_gib:.2f} GiB（另需預留 3 GiB）。'
)
print(f'選擇 {MODEL_VARIANT}/{QUANT}｜去噪權重約 {QUANT_SIZES_GIB[QUANT]:.2f} GiB｜'
      f'尚需下載約 {missing_gib:.2f} GiB｜/tmp 可用 {tmp_free_gib:.1f} GiB')

'''
Warn when high-bit quantizations will require greater GPU/CPU offload.
高精度量化需要較多 GPU／CPU 卸載時，事先提醒使用者。
'''
if QUANT in ('Q6_K', 'Q8_0'):
    print('注意：Q6／Q8 可能需要大量 CPU Offload，速度較慢且有記憶體不足風險。')

for repo, filename, folder in models:
    print('模型：', Path(filename).name, flush=True)
    source = next((path for path in GGUF_CACHE_PATHS
                   if folder == 'diffusion_models' and path.is_file()
                   and 0.97 * QUANT_SIZES_GIB[QUANT] <= path.stat().st_size / 1024**3 <= 1.03 * QUANT_SIZES_GIB[QUANT]), None)

    '''
    Download files with Hub-managed validation when no complete cache exists.
    沒有完整快取時，交由 Hugging Face Hub 管理下載及檔案驗證。
    '''
    if source is None:
        local_file = STORE / folder / Path(filename).name
        expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
        invalid_cache = local_file.is_file() and not (0.97 * expected_gib <= local_file.stat().st_size / 1024**3 <= 1.03 * expected_gib)

        '''
        Keep models under /tmp without copying gigabytes into 設定的工作.
        模型實體仍放在 /tmp，避免複製大型檔案到 設定的工作。
        '''
        source = Path(hf_hub_download(repo_id=repo, filename=filename,
                                      local_dir=STORE / folder, force_download=invalid_cache))
    expected_gib = QUANT_SIZES_GIB[QUANT] if folder == 'diffusion_models' else AUX_SIZES_GIB[Path(filename).name]
    assert 0.97 * expected_gib <= source.stat().st_size / 1024**3 <= 1.03 * expected_gib, f'模型檔案大小異常：{source}'
    target = COMFY / 'models' / folder / source.name
    target.parent.mkdir(parents=True, exist_ok=True)

    '''
    Keep an already-correct symbolic link to avoid unnecessary service restarts.
    已指向正確權重的符號連結直接沿用，避免不必要的服務重新啟動。
    '''
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):
        target.unlink()
    if not target.exists():
        target.symlink_to(source.resolve())
        H3_RESTART_REQUIRED = True
    assert target.samefile(source), f'模型檔案衝突：{target} 與預期來源 {source} 不同。請先檢查舊檔。'
    assert target.is_file(), f'模型連結失敗：{target}'
    print('ComfyUI 路徑：', target)

obsolete_link = COMFY / 'models' / 'unet_gguf' / DIT_NAME
if obsolete_link.is_symlink():
    obsolete_link.unlink()

'''
Confirm all requested model paths can be opened.
確認所有所需模型檔案都可正常存取。
'''
assert all((COMFY / 'models' / folder / Path(filename).name).is_file() for _, filename, folder in models)
print(f'H3 {MODEL_VARIANT}/{QUANT}、4B ClipProj、Video VAE、Audio VAE 已就緒。')
H3_MODEL_CHOICE = (MODEL_VARIANT, QUANT)

'''
Remind users to revalidate the running loader after changing quantizations.
提醒切換量化格式後，先更新目前 ComfyUI 的模型選單。
'''
print('切換量化格式後：請重新執行第 4 節，再執行第 11 節生成。')
print(f'✓ 第 3 節完成：{MODEL_VARIANT}/{QUANT} 模型與相關權重已備妥（/tmp）。')


4. GPU 記憶體與 CPU Offload 設定


In [ ]:
'''
Require the teacher settings cell before applying ComfyUI GPU configuration.
套用 ComfyUI GPU 設定前，確認已執行教師共用設定 Cell。
'''
assert all(name in globals() for name in ('GPU0_MODEL_GIB', 'GPU1_MODEL_GIB', 'TEXT_MODE', 'TEXT_GPU', 'NODE_CACHE_MODE')), '請先執行「教師共用設定」Cell。'
gpu0_capacity = torch.cuda.get_device_properties(MODEL_GPU_ID).total_memory / 1024**3
gpu1_capacity = torch.cuda.get_device_properties(AUX_GPU_ID).total_memory / 1024**3

'''
Reject values that exceed the available GPU memory.
避免設定超過 GPU 實際容量的數值。
'''
assert 0 < GPU0_MODEL_GIB <= gpu0_capacity, f'GPU0_MODEL_GIB 必須介於 0 和 {gpu0_capacity:.2f} GiB 之間。'
assert 0 < GPU1_MODEL_GIB <= gpu1_capacity, f'GPU1_MODEL_GIB 必須介於 0 和 {gpu1_capacity:.2f} GiB 之間。'
assert TEXT_MODE in ('dynamic', 'streaming', 'resident'), 'TEXT_MODE 設定無效。'

'''
Reserve the difference for GPU 0 so ComfyUI targets the chosen overall budget.
將 GPU 0 的容量差額保留，讓 ComfyUI 以指定的整體預算運作。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

assert NODE_CACHE_MODE in ('classic', 'none')
H3_DESIRED_CONFIG = (MODEL_GPU_ID, AUX_GPU_ID, GPU0_RESERVE_GIB, NODE_CACHE_MODE)

'''
Clarify that per-model residency is not a hard limit.
明確提醒：這不是單一模型的精確權重常駐限制。
'''
print('GPU 0：ComfyUI Dynamic VRAM / CPU Offload（權重、運算暫存及其它元件共用預算）。')
print(f'GPU {AUX_GPU_ID}：ClipProj {TEXT_MODE}，{GPU1_MODEL_GIB:.2f} GiB 為監控警示值，並非硬上限。')
print(f'✓ 第 4 節完成：GPU {MODEL_GPU_ID} 管理預算約 {GPU0_MODEL_GIB:.2f} GiB；保留 {GPU0_RESERVE_GIB:.2f} GB。')


5. 啟動本機 ComfyUI 與相依性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time
LOG = ROOT / 'minimax_h3_comfyui.log'
BASE = f'http://127.0.0.1:{PORT}'

'''
Read server readiness without requiring a public tunnel.
不開放公開連線即可檢查伺服器狀態。
'''
def ready():
    '''
    Handle an unavailable local service safely.
    安全處理尚未啟動的服務。
    '''
    try:
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:
        return False

'''
Read the ComfyUI loader's actual model dropdown values.
讀取 ComfyUI GGUF Loader 真正提供的模型選項。
'''
def listed_ggufs():
    '''
    Retrieve the registered GGUF model names from the local service.
    從本機服務取得已註冊的 GGUF 模型檔名。
    '''
    payload = requests.get(f'{BASE}/object_info/UnetLoaderGGUF', timeout=30).json()
    info = payload.get('UnetLoaderGGUF', {})
    spec = info.get('input', {}).get('required', {}).get('unet_name')
    return spec[0] if isinstance(spec, list) and spec and isinstance(spec[0], list) else []

need_restart = False
if ready():
    need_restart = (
        globals().get('H3_RESTART_REQUIRED', False)
        or DIT_NAME not in listed_ggufs()
        or globals().get('H3_ACTIVE_MODEL') != (MODEL_VARIANT, QUANT)
        or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG
    )

'''
Never terminate services launched by another notebook or process.
不任意終止其它 Notebook 或行程啟動的 ComfyUI。
'''
if need_restart:
    '''
    Only restart a service for which this kernel holds the process handle.
    只有目前 Kernel 持有行程控制權時才能自動重新啟動。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, (
        '服務需要更新記憶體設定，但不是本 Notebook 可管理的行程。'
        f'請先停止連接埠 {PORT} 的舊服務，再執行本 Cell。'
    )
    H3_PROC.terminate()
    try:
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:
        '''
        Handle a service that does not respond to graceful shutdown.
        處理無法正常關閉的服務。
        '''
        H3_PROC.kill()
        H3_PROC.wait(timeout=10)
    print('重新啟動 ComfyUI：模型索引、節點版本或 GPU 0 記憶體預算有變更。')

if not ready():
    command = [
        sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT), '--cuda-device', str(MODEL_GPU_ID),
        '--enable-dynamic-vram', '--reserve-vram', str(GPU0_RESERVE_GIB),
        '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
        '--disable-pinned-memory',
    ]

    '''
    Keep the local service log available for debugging without blocking Kaggle.
    保留服務 Log 方便除錯，且不讓 Notebook 卡住。
    '''
    with LOG.open('w', encoding='utf-8') as stream:
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)
    for attempt in range(180):
        '''
        Report the original service error rather than waiting indefinitely.
        發生啟動錯誤時直接輸出 Log，避免無限等待。
        '''
        if H3_PROC.poll() is not None:
            raise RuntimeError(LOG.read_text(errors='replace')[-4000:])
        if ready():
            break
        time.sleep(2)
    assert ready(), LOG.read_text(errors='replace')[-4000:]

H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Get the exact input schemas registered by the running ComfyUI process.
從目前執行中的 ComfyUI 讀取真正註冊的節點輸入規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
REQUIRED_NODES = (
    'ClipProjLoader', 'UnetLoaderGGUF', 'MiniMaxH3SigmaShift',
    'VAELoader', 'MiniMaxH3ImageToVideo', 'BasicGuider', 'KSamplerSelect',
    'BasicScheduler', 'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode',
    'VAEDecodeAudio', 'CreateVideo', 'SaveVideo', 'LoadImage',
    'MiniMaxH3ReferenceToVideo',
)
missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show a clear explanation for a failed or incomplete ComfyUI installation.
若 ComfyUI 安裝不完整，直接指出缺少的節點。
'''
assert not missing, f'ComfyUI 缺少節點：{missing}。請檢查 {LOG} 並重新執行安裝與啟動。'

'''
Read a dropdown's allowed values from ComfyUI's live schema.
從 ComfyUI 正在使用的規格讀取下拉選單允許值。
'''
def choices(node_name, input_name):
    '''
    Support both required and optional node inputs.
    同時支援必填與選填的節點輸入。
    '''
    groups = NODE_INFO[node_name].get('input', {})
    for group in ('required', 'optional'):
        spec = groups.get(group, {}).get(input_name)

        '''
        Return None for numeric/string inputs that are not dropdown lists.
        如果不是下拉式選單，就不視為有限選項。
        '''
        if spec is not None:
            if isinstance(spec, (list, tuple)) and spec:
                if isinstance(spec[0], list):
                    return spec[0]
                if len(spec) > 1 and isinstance(spec[1], dict):
                    for key in ('choices', 'options'):
                        if isinstance(spec[1].get(key), list):
                            return spec[1][key]
            return None
    raise RuntimeError(f'節點 {node_name} 沒有預期的輸入 {input_name}；可能是節點版本不相容.')

'''
Validate dropdown values without crashing if a node exposes only a scalar input.
驗證下拉選項；如果節點只提供一般輸入型別，就不要因此發生例外。
'''
def check_choice(node_name, input_name, value, fallback=None):
    '''
    Read the node's current explicit choice list if present.
    有明確選項清單時優先讀取目前節點的清單。
    '''
    allowed = choices(node_name, input_name)
    if allowed is None and fallback and fallback[0] in NODE_INFO:
        allowed = choices(*fallback)
    if allowed is None:
        print(f'預檢提示：{node_name}.{input_name} 沒有選項清單，將交由 ComfyUI 驗證。')
    else:
        assert value in allowed, f'{node_name}.{input_name} 不支援 {value!r}；可用選項：{allowed}'

assert DIT_NAME in (choices('UnetLoaderGGUF', 'unet_name') or []), f'GGUF Loader 沒有辨識 {DIT_NAME}。'
assert 'qwen3vl_4b_fp8_scaled.safetensors' in (choices('ClipProjLoader', 'clip_name') or []), 'ClipProj Loader 沒有辨識 4B Encoder。'

'''
Verify that the correct projection matrix is registered, not merely downloaded.
確認正確的投影矩陣已註冊，而不只是檔案下載成功。
'''
assert 'mmh3-4b-ClipProj-v3.1-mlp.safetensors' in (choices('ClipProjLoader', 'projection') or []), 'ClipProj Loader 沒有辨識 v3.1 投影矩陣。'
assert 'auto' in (choices('ClipProjLoader', 'type') or []), 'ClipProj Loader 不支援 type=auto，請更新節點。'
assert 'krea2' in (choices('ClipProjLoader', 'type') or []), '目前 ComfyUI 尚未提供 krea2／Qwen3-VL 4B 編碼器載入支援，請更新 ComfyUI。'

'''
Verify the supported video and audio VAE filenames.
確認影片與音訊 VAE 都出現在實際模型選單中。
'''
for vae_name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):
    assert vae_name in (choices('VAELoader', 'vae_name') or []), f'ComfyUI 沒有辨識 VAE：{vae_name}'

print('依賴檢查通過：FL2VA、Ref2VA、GGUF、4B Encoder、ClipProj v3.1、Video/Audio VAE。')
H3_ACTIVE_QUANT = QUANT

'''
Remember the model family as well as its quantization.
同時記錄模型種類及量化格式，避免混用 FL2VA／Ref2VA。
'''
H3_ACTIVE_MODEL = (MODEL_VARIANT, QUANT)
H3_RESTART_REQUIRED = False
print(f'本機 MiniMax H3 {MODEL_VARIANT}/{QUANT} 已就緒（GGUF 模型選單驗證通過）。')

'''
Confirm that all service and model checks finished without an exception.
所有服務及模型檢查通過後，明確顯示本格完成。
'''
print(f'✓ 第 5 節完成：ComfyUI 服務與模型預檢完成；GPU 0 保留 {GPU0_RESERVE_GIB:.2f} GB。')


6. T2V：Prompt 與參數設定


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()
T2V_WIDTH, T2V_HEIGHT = 384, 224
T2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
T2V_FPS = 24

'''
Choose sampling iterations for the original model.
設定原始模型的採樣步數。
'''
T2V_STEPS = 20
T2V_AUDIO_SHIFT = 6.0
T2V_SEED = 42

'''
Set the T2V output filename without overwriting earlier runs.
設定文生影片輸出檔名；若同名會自動編號。
'''
T2V_OUTPUT = 'minimax_h3_t2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           seconds=T2V_SECONDS, fps=T2V_FPS, steps=T2V_STEPS, seed=T2V_SEED,
                           audio_shift=T2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Provide explicit feedback after configuring the task.
完成任務設定後顯示明確提示。
'''
TASK = 't2v'

print(f'✓ 第 6 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {QUANT} | Audio Shift {T2V_AUDIO_SHIFT} | {T2V_OUTPUT}')


7. I2V：Prompt、起始圖片與參數設定


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
I2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
I2V_FPS = 24
I2V_STEPS = 20
I2V_AUDIO_SHIFT = 6.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定本次任務的種子；-1 表示每次重新抽取。量化設定沿用第 1 節的 QUANT。
'''
I2V_SEED = 42
I2V_OUTPUT = 'minimax_h3_i2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the first-frame conditioning input and task parameters.
登記起始影格及圖生影片參數。
'''
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           seconds=I2V_SECONDS, fps=I2V_FPS, steps=I2V_STEPS, seed=I2V_SEED,
                           audio_shift=I2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'i2v'

print(f'✓ 第 7 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {QUANT} | Audio Shift {I2V_AUDIO_SHIFT} | 圖片：{I2V_FIRST_IMAGE}')


8. L2VA：最後影格生成影片


In [ ]:
'''
Describe how the uploaded last frame should animate.
描述上傳的最後一張圖片要如何轉化為動態影片。
'''
L2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
L2V_LAST_IMAGE = str(ROOT / 'last_frame.png')
L2V_WIDTH, L2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定最後影格生成影片的目標秒數。
'''
L2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
L2V_FPS = 24
L2V_STEPS = 20
L2V_AUDIO_SHIFT = 6.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定本次任務的種子；-1 表示每次重新抽取。量化設定沿用第 1 節的 QUANT。
'''
L2V_SEED = 42
L2V_OUTPUT = 'minimax_h3_l2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register the last-frame conditioning input and task parameters.
登記最後影格及圖生影片參數。
'''
TASK_CONFIGS['l2v'] = dict(prompt=L2V_PROMPT, width=L2V_WIDTH, height=L2V_HEIGHT,
                           seconds=L2V_SECONDS, fps=L2V_FPS, steps=L2V_STEPS, seed=L2V_SEED,
                           audio_shift=L2V_AUDIO_SHIFT, quant=QUANT,
                           output_name=L2V_OUTPUT, first_image='', last_image=L2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'l2v'

print(f'✓ 第 8 節完成：L2V | {L2V_WIDTH}×{L2V_HEIGHT} | {L2V_SECONDS} 秒 | {QUANT} | Audio Shift {L2V_AUDIO_SHIFT} | 圖片：{L2V_LAST_IMAGE}')


9. FLF2V：Prompt、首尾圖片與參數設定


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()
FLF2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
FLF2V_LAST_IMAGE = str(ROOT / 'last_frame.png')

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224
FLF2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
FLF2V_FPS = 24
FLF2V_STEPS = 20

'''
Set the audio sigma shift for this generation mode (ComfyUI default: 3.0).
設定這個生成模式的音訊 Sigma Shift（ComfyUI 預設：3.0）。
'''
FLF2V_AUDIO_SHIFT = 6.0
FLF2V_SEED = 42
FLF2V_OUTPUT = 'minimax_h3_flf2v.mp4'

'''
Keep previously configured modes unchanged.
保留先前其它生成模式的設定。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             seconds=FLF2V_SECONDS, fps=FLF2V_FPS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             audio_shift=FLF2V_AUDIO_SHIFT, quant=QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'flf2v'

print(f'✓ 第 9 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {QUANT} | Audio Shift {FLF2V_AUDIO_SHIFT} | 參考圖 2 張')


10. R2V：Prompt、圖片／影片／音訊參考


In [ ]:
'''
Describe how multiple reference images define the subject and environment.
使用多張參考圖片指定人物外觀、服裝及環境。
'''
R2V_PROMPT = '''
<Picture 1> defines the subject's face, hairstyle, and identity.
Additional pictures, when provided, help specify the outfit and environment.
Create a cinematic video of the subject walking naturally through the scene.
Preserve identity, clothing, and scene details from the reference pictures.
Natural motion, soft daylight, realistic ambient sounds.
'''.strip()
R2V_REF_IMAGES = []
R2V_REF_VIDEOS = []  # Example: ['/home/your_user/ai_models/clip.mp4']

'''
Optional standalone reference audio files (up to three).
Ref2VA 可選的獨立音訊路徑（最多三段，依 <Audio N> 編號排序）。
'''
R2V_REF_AUDIOS = []  # Example: ['/home/your_user/ai_models/voice.wav']
R2V_VIDEO_AUDIO = False  # True, False
R2V_REF_IMAGE_SIZE = 'match'

'''
Select the R2V-specific ClipProj mode: resident, dynamic, or streaming. resident = prioritize vision-encoder stability; dynamic = lower residency; streaming = page weights. Available choices: resident, dynamic, streaming.
R2V 專用 ClipProj 模式可選 resident、dynamic、streaming。 resident 優先保持影像編碼器常駐；dynamic 動態卸載；streaming 使用分段載入。 可選值：resident、dynamic、streaming。
'''
R2V_TEXT_MODE = TEXT_MODE
R2V_WIDTH, R2V_HEIGHT = 384, 224
R2V_SECONDS = 10

'''
Control video FPS for both the notebook and its remote API.
同時控制 Notebook 與遠端 API 的影片影格率。
'''
R2V_FPS = 24

'''
Set the diffusion sampling steps for the normal Ref2VA model.
設定正常版 Ref2VA 模型的採樣步數。
'''
R2V_STEPS = 20
R2V_AUDIO_SHIFT = 3.0

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
R2V_SEED = 42
R2V_OUTPUT = 'minimax_h3_r2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Save the reference media and settings without downloading extra models yet.
儲存參考圖與參數，不在此 Cell 下載額外權重。
'''
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                            seconds=R2V_SECONDS, fps=R2V_FPS, steps=R2V_STEPS, seed=R2V_SEED,
                            audio_shift=R2V_AUDIO_SHIFT, quant=QUANT,
                            output_name=R2V_OUTPUT, first_image='', last_image='',
                            ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE,
                            ref_videos=R2V_REF_VIDEOS, ref_audios=R2V_REF_AUDIOS,
                            video_audio=R2V_VIDEO_AUDIO, text_mode=R2V_TEXT_MODE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'r2v'

print(f'✓ 第 10 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {R2V_SECONDS} 秒 | '
      f'{QUANT} | {len(R2V_REF_IMAGES)} 張圖片；若尚未下載 R2V 權重，請重新執行第 2、4 節。')

import ipywidgets as widgets
from pathlib import Path
from PIL import Image, ImageOps
from io import BytesIO
from uuid import uuid4
from html import escape
from IPython.display import display, clear_output
import shutil
_REF_ROOT = R2V_REF_DIR
_REF_ROOT.mkdir(parents=True, exist_ok=True)
_REF_MAX = 9
_REF_FILES = sorted(p for p in _REF_ROOT.iterdir() if p.is_file() and p.suffix.lower() in ('.png','.jpg','.jpeg','.webp'))
if len(_REF_FILES) > _REF_MAX or any(p.stem != f'{i:03d}' for i,p in enumerate(_REF_FILES,1)):
    raise RuntimeError('參考圖編號不連續；請清理舊資料夾再執行上傳 Cell。')

'''
Synchronize the numbered paths with the generation configuration.
將最新圖片清單同步到生成工作所使用的設定。
'''
def _ref_sync(paths):
    global R2V_REF_IMAGES
    R2V_REF_IMAGES = [str(path) for path in paths]
    TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)

'''
Replace all reference images only after the new upload is fully validated.
新圖片全部通過驗證後才取代上一批；選得更少時也會清除多餘舊檔。
'''
def _ref_update(change):
    items = change['new']
    if not items:
        return
    if isinstance(items, dict):
        items = [dict(name=name, **item) for name, item in items.items()]
    stage = _REF_ROOT.with_name(_REF_ROOT.name + '_stage_' + uuid4().hex)
    stage.mkdir(parents=True)
    backup = _REF_ROOT.with_name(_REF_ROOT.name + '_backup_' + uuid4().hex)
    try:
        if not 1 <= len(items) <= _REF_MAX:
            raise ValueError(f'一次請選擇 1～{_REF_MAX} 張圖片。')
        for index, item in enumerate(items, 1):
            filename = Path(item['name']).name
            suffix = Path(filename).suffix.lower()
            if suffix not in ('.png','.jpg','.jpeg','.webp'):
                raise ValueError(f'不支援的圖片格式：{filename}')
            blob = bytes(item['content'])
            if len(blob) > 20 * 1024 * 1024:
                raise ValueError(f'單張圖片超過 20 MiB：{filename}')
            with Image.open(BytesIO(blob)) as photo:
                photo.verify()
            (stage / f'{index:03d}{suffix}').write_bytes(blob)
        _REF_ROOT.rename(backup)
        try:
            stage.rename(_REF_ROOT)
        except Exception:
            backup.rename(_REF_ROOT)
            raise
        shutil.rmtree(backup)
        current = sorted(_REF_ROOT.iterdir())
        _ref_sync(current)
        print('最新參考圖：', ', '.join(f'{p.name} = <Picture {i}>' for i,p in enumerate(current,1)))
    except Exception as error:
        shutil.rmtree(stage, ignore_errors=True)
        print('上傳未生效，保留上一批參考圖：', error)
    finally:
        _REF_PANEL.children = (_ref_new(),)

'''
Reset the browser picker so selecting the same filenames works again.
每次上傳後重建選檔元件，確保可以重新選相同檔名。
'''
def _ref_new():
    picker = widgets.FileUpload(accept='image/*', multiple=True, description='多選參考圖片')
    picker.observe(_ref_update, names='value')
    return picker

_REF_PANEL = widgets.VBox()
_REF_PANEL.children = (_ref_new(),)
_ref_sync(_REF_FILES)
display(_REF_PANEL)
print('目前圖片：', ', '.join(p.name for p in _REF_FILES) or '尚未上傳')
print('✓ 第 10 節完成：R2V 設定與圖片上傳介面已就緒（尚未生成）。')


11. 生成影片（沿用最後執行的模式 Cell，顯示即時步數進度）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import json
    import shutil
    import uuid
    import re
    import math
    import websocket
    from tqdm.auto import tqdm
    TASK = globals().get('TASK', 't2v')
    assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'
    config = TASK_CONFIGS[TASK]

    '''
    Keep one submission flow shared across all four tasks.
    四種生成模式共用同一份工作流程提交邏輯。
    '''
    MODE = TASK
    PROMPT = config['prompt']
    WIDTH, HEIGHT = config['width'], config['height']
    SECONDS, STEPS = config['seconds'], config['steps']

    '''
    Apply the task-specific output frame rate to generation and encoding.
    將各模式的輸出影格率同步用於生成影格數與影片編碼。
    '''
    FPS = config['fps']
    SEED, OUTPUT_NAME = config['seed'], config['output_name']

    '''
    Resolve task-specific reference images without mixing modes.
    依據所選任務使用對應參考圖片，避免不同模式混用。
    '''
    FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']
    AUDIO_SHIFT = config['audio_shift']
    QUANT = config['quant']
    assert QUANT in QUANT_SIZES_GIB, f'不支援的 QUANT：{QUANT}'
    REF_IMAGES = config.get('ref_images', [])

    '''
    Select optional reference videos and standalone audio.
    取得選用的參考影片與獨立音訊。
    '''
    REF_VIDEOS = list(config.get('ref_videos', []))
    REF_AUDIOS = list(config.get('ref_audios', []))
    VIDEO_AUDIO = bool(config.get('video_audio', False))
    REF_IMAGE_SIZE = config.get('ref_image_size', 'match')
    ACTIVE_TEXT_MODE = config.get('text_mode', TEXT_MODE)
    assert 0.01 <= AUDIO_SHIFT <= 100.0, 'AUDIO_SHIFT 必須介於 0.01 與 100.0 之間。'

    '''
    Ensure edited GPU 0 memory budgets have been applied to the running service.
    確認調整過的 GPU 0 記憶體預算已套用到目前服務。
    '''
    assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'
    EXPECTED_VARIANT = 'ref2va_pruned' if MODE == 'r2v' else 'fl2va_pruned'
    assert globals().get('H3_MODEL_CHOICE') == (EXPECTED_VARIANT, QUANT) and globals().get('H3_ACTIVE_MODEL') == (EXPECTED_VARIANT, QUANT), (
        f'目前選擇 {QUANT}，但尚未完成下載或服務預檢。請先執行第 2 節與第 4 節。'
    )
    assert DIT_NAME == f'minimax_h3_{EXPECTED_VARIANT}-{QUANT}.gguf', '模型檔名與任務種類／QUANT 不一致，請重跑第 2 節。'
    LAST_OUTPUT = None

    '''
    Confirm the existing local service is still running.
    確認先前啟動的本機推論服務仍正常運作。
    '''
    assert ready(), 'ComfyUI 已中斷，請先重新執行第 4 節。'
    assert MODE in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
        '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
    )
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
        'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
    )
    assert isinstance(FPS, int) and not isinstance(FPS, bool) and 1 <= FPS <= 60, 'FPS 必須是 1～60 的整數。'
    FRAMES = max(5, 17 * math.ceil((SECONDS * FPS - 5) / 17) + 5)

    '''
    Respect the underlying ComfyUI node's frame limit, not an arbitrary notebook duration.
    遵守 ComfyUI 節點真正的影格數上限，不另外設定 Notebook 的秒數上限。
    '''
    assert FRAMES <= 3600, f'要求 {FRAMES} frames，超出 MiniMax H3 節點上限 3600；請縮短 SECONDS。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')

    '''
    Build a unique ComfyUI output prefix from the user filename.
    根據指定檔名建立不重複的 ComfyUI 輸出前綴。
    '''
    output_stem = Path(OUTPUT_NAME).stem
    run_name = f'{output_stem}_{EXPECTED_VARIANT}_{QUANT}_{uuid.uuid4().hex[:8]}'
    graph = {
        '1': {'class_type': 'ClipProjLoader', 'inputs': {

            'clip_name': 'qwen3vl_4b_fp8_scaled.safetensors', 'type': 'auto',
            'projection': 'mmh3-4b-ClipProj-v3.1-mlp.safetensors',
            'device': TEXT_GPU, 'mode': ACTIVE_TEXT_MODE}},
        '2': {'class_type': 'UnetLoaderGGUF', 'inputs': {'unet_name': DIT_NAME}},
        '3': {'class_type': 'MiniMaxH3SigmaShift', 'inputs': {
            'model': ['2', 0], 'shift_video': 12.0, 'shift_audio': AUDIO_SHIFT}},
        '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors'}},
        '5': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors'}},
        '6': {'class_type': 'MiniMaxH3ImageToVideo', 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
            'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
        '7': {'class_type': 'BasicGuider', 'inputs': {
            'model': ['3', 0], 'conditioning': ['6', 0]}},
        '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'res_multistep'}},
        '9': {'class_type': 'BasicScheduler', 'inputs': {
            'model': ['3', 0], 'scheduler': 'simple', 'steps': STEPS, 'denoise': 1.0}},
        '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
        '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['10', 0], 'guider': ['7', 0],
            'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
        '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
        '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
        '14': {'class_type': 'CreateVideo', 'inputs': {
            'images': ['12', 0], 'audio': ['13', 0], 'fps': FPS}},
        '15': {'class_type': 'SaveVideo', 'inputs': {
            'video': ['14', 0], 'filename_prefix': run_name,
            'format': 'auto'}},
    }
    if MODE == 'r2v':
        assert isinstance(REF_IMAGES, (tuple, list)) and 0 <= len(REF_IMAGES) <= 9, 'R2V 圖片最多 9 張，也可以只提供影片或音訊。'
        assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'
        picture_numbers = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, flags=re.IGNORECASE)}

        '''
        Refuse to submit a prompt referencing an absent image.
        若 Prompt 指向不存在的圖片，避免送出無效的參考條件。
        '''
        assert all(1 <= number <= len(REF_IMAGES) for number in picture_numbers), (
            f'R2V_PROMPT 引用了不存在的 <Picture N>：{sorted(picture_numbers)}；目前只有 {len(REF_IMAGES)} 張。'
        )
        check_choice('MiniMaxH3ReferenceToVideo', 'ref_image_size', REF_IMAGE_SIZE)
        graph['6'] = {'class_type': 'MiniMaxH3ReferenceToVideo', 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'audio_vae': ['5', 0],
            'prompt': PROMPT, 'width': WIDTH, 'height': HEIGHT, 'length': FRAMES,
            'ref_image_size': REF_IMAGE_SIZE}}
        for image_index, image_path in enumerate(REF_IMAGES):
            source = Path(image_path)
            assert source.is_file(), f'R2V 找不到第 {image_index + 1} 張參考圖片：{source}'
            assert source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp', '.bmp'), f'不支援的參考圖片格式：{source.suffix}'
            node_id = str(16 + image_index)

            '''
            Copy files into ComfyUI's private input directory with unique names.
            將參考圖片複製到 ComfyUI 私有輸入資料夾，並使用不重複檔名。
            '''
            destination = COMFY / 'input' / f'r2v_{image_index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][f'ref_images.ref_image_{image_index}'] = [node_id, 0]

    '''
    Validate reference media counts and numbered prompt placeholders.
    驗證圖片、影片、音訊張數與 Prompt 的參考編號。
    '''
    if MODE == 'r2v':
        assert len(REF_IMAGES) <= 9 and len(REF_VIDEOS) <= 3 and len(REF_AUDIOS) <= 3, 'R2V 最多 9 張圖片、3 段影片、3 段獨立音訊。'
        assert REF_IMAGES or REF_VIDEOS, 'R2V 至少需要一張參考圖片或一段參考影片；音訊可作為附加條件。'
        assert len(REF_IMAGES) + len(REF_VIDEOS) + len(REF_AUDIOS) <= 12, 'R2V 參考素材合計最多 12 個檔案。'
        tags = {'Picture': len(REF_IMAGES), 'Video': len(REF_VIDEOS),
                'Audio': len(REF_AUDIOS) + (len(REF_VIDEOS) if VIDEO_AUDIO else 0)}
        for label, count in tags.items():
            numbers = [int(n) for n in re.findall(r'<%s\s+(\d+)>' % label, PROMPT, re.I)]
            assert all(1 <= n <= count for n in numbers), f'<{label} N> 編號超出實際上傳數量：{numbers}。'

        '''
        Read reference videos using native ComfyUI video nodes, without extra pip packages.
        使用 ComfyUI 原生影片節點載入參考影格與選用的影片音軌。
        '''
        for index, filename in enumerate(REF_VIDEOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.mp4', '.mov', '.webm'), f'參考影片不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_video_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(120 + index * 2)
            parts_id = str(121 + index * 2)
            graph[node_id] = {'class_type': 'LoadVideo', 'inputs': {'file': destination.name}}
            graph[parts_id] = {'class_type': 'GetVideoComponents', 'inputs': {'video': [node_id, 0]}}
            graph['6']['inputs'][f'ref_videos.ref_video_{index}'] = [parts_id, 0]
            if VIDEO_AUDIO:
                graph['6']['inputs'][f'ref_video_audios.ref_video_audio_{index}'] = [parts_id, 1]
        for index, filename in enumerate(REF_AUDIOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'), f'參考音訊不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_audio_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(140 + index)
            graph[node_id] = {'class_type': 'LoadAudio', 'inputs': {'audio': destination.name}}
            graph['6']['inputs'][f'ref_audios.ref_audio_{index}'] = [node_id, 0]
    for mode, image_path, node_id, input_name in [
        ('i2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', LAST_IMAGE, '17', 'last_frame'),
        ('l2v', LAST_IMAGE, '18', 'last_frame'),
    ]:
        if MODE == mode:
            '''
            Require an existing image file for the selected mode.
            確認所選模式的圖片檔案確實存在。
            '''
            source = Path(image_path)
            assert source.is_file(), f'找不到參考圖片：{source}'
            destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)

            '''
            Register the image loader in the API graph.
            將圖片載入節點加入 API Workflow。
            '''
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][input_name] = [node_id, 0]
    NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
    check_choice('ClipProjLoader', 'device', TEXT_GPU)
    check_choice('ClipProjLoader', 'mode', ACTIVE_TEXT_MODE)

    '''
    Verify the H3 sampler against the specialized node or the core KSampler options.
    優先用目前採樣器選單驗證；必要時參照核心 KSampler 選項。
    '''
    check_choice('KSamplerSelect', 'sampler_name', 'res_multistep', ('KSampler', 'sampler_name'))
    check_choice('BasicScheduler', 'scheduler', 'simple', ('KSampler', 'scheduler'))
    if MODE in ('i2v', 'l2v', 'flf2v'):
        frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})
        expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else (('last_frame',) if MODE == 'l2v' else ('first_frame',))

        '''
        Reject an incompatible core node implementation before generation.
        如果核心節點不支援這些欄位，在生成之前先停止。
        '''
        assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'
    if MODE == 'r2v':
        ref_inputs = NODE_INFO['MiniMaxH3ReferenceToVideo'].get('input', {})
        ref_fields = set(ref_inputs.get('required', {})) | set(ref_inputs.get('optional', {}))
        assert 'ref_images' in ref_fields or any(name.startswith('ref_images.ref_image_') for name in ref_fields), (
            'MiniMaxH3ReferenceToVideo 缺少 ref_images 輸入，請更新 ComfyUI。'
        )

    '''
    Confirm the installed ComfyUI version supports all reference-media loader nodes.
    確認目前 ComfyUI 具備參考影片與音訊所需的原生節點。
    '''
    if MODE == 'r2v':
        required_nodes = {'LoadVideo', 'GetVideoComponents'} if REF_VIDEOS else set()
        if REF_AUDIOS:
            required_nodes.add('LoadAudio')
        missing_nodes = required_nodes - set(NODE_INFO)
        assert not missing_nodes, f'ComfyUI 缺少參考媒體節點：{sorted(missing_nodes)}，請更新 ComfyUI。'
    for node_id, node in graph.items():
        '''
        Examine each declared input for possible node-reference links.
        檢查節點輸入中是否包含連到其它節點的參照。
        '''
        for value in node['inputs'].values():
            if isinstance(value, list) and len(value) == 2:
                assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'
    for node_id, node in graph.items():
        schema = NODE_INFO[node['class_type']].get('input', {})
        accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))
        unknown = set(node['inputs']) - accepted

        '''
        Allow dynamic Ref2VA image slots declared by the live autogrow parent.
        當節點宣告 ref_images 動態欄位時，允許自動展開的圖片插槽。
        '''
        if node['class_type'] == 'MiniMaxH3ReferenceToVideo' and 'ref_images' in accepted:
            unknown -= {f'ref_images.ref_image_{i}' for i in range(9)}
            if 'ref_videos' in accepted:
                unknown -= {f'ref_videos.ref_video_{i}' for i in range(3)}
            if 'ref_video_audios' in accepted:
                unknown -= {f'ref_video_audios.ref_video_audio_{i}' for i in range(3)}
            if 'ref_audios' in accepted:
                unknown -= {f'ref_audios.ref_audio_{i}' for i in range(3)}
        assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"
    print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | {EXPECTED_VARIANT}/{QUANT} | Audio Shift {AUDIO_SHIFT}', flush=True)
    client_id = uuid.uuid4().hex

    '''
    Track WebSocket availability separately from workflow success.
    個別追蹤 WebSocket 可用性，不影響實際生成流程。
    '''
    ws = None
    try:
        ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)
        ws.settimeout(2)

    except (OSError, websocket.WebSocketException):
        '''
        Mark event updates as unavailable and retain an elapsed-time display.
        停用事件更新，改成單一進度列顯示經過時間。
        '''
        ws = None
        print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)
    response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)
    if not response.ok:
        if ws is not None:
            ws.close()
        raise RuntimeError(response.text[:5000])
    prompt_id = response.json()['prompt_id']

    '''
    Start timing the actual queued job.
    開始計算已提交生成工作的耗時。
    '''
    start = time.monotonic()
    pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)
    item = None
    last_history_check = 0.0
    phases = {'1': '文字編碼器', '2': '載入 GGUF', '6': '建立影片條件',
              '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
              '14': '合成影片', '15': '儲存影片'}

    '''
    Wait until ComfyUI reports a final history record for this task.
    持續等待 ComfyUI 回傳此任務的最終執行紀錄。
    '''
    try:
        '''
        Keep processing until success, an explicit error, or timeout.
        持續處理直到成功、明確錯誤或逾時。
        '''
        while True:
            '''
            Measure real wall-clock time instead of estimating model progress.
            計算真實經過時間，不將時間誤當模型百分比。
            '''
            elapsed = time.monotonic() - start
            if elapsed > 7200:
                raise TimeoutError(f'生成逾時；prompt_id={prompt_id}，可查詢 {BASE}/history/{prompt_id}')
            message = None
            if ws is not None:
                try:
                    message = ws.recv()

                except websocket.WebSocketTimeoutException:
                    pass

                except (OSError, websocket.WebSocketException):
                    '''
                    Release the unusable client connection.
                    關閉失效的 WebSocket 連線。
                    '''
                    ws.close()
                    ws = None

            '''
            Only JSON text frames contain progress data; preview images are binary.
            只有 JSON 文字事件含進度資料；預覽圖片為二進位資料。
            '''
            if isinstance(message, str):
                event = json.loads(message)

                '''
                Read the event payload without assuming optional keys exist.
                讀取事件資料，並避免假設選填欄位一定存在。
                '''
                data = event.get('data') or {}
                if data.get('prompt_id') == prompt_id:
                    if event.get('type') == 'executing' and data.get('node'):
                        pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)
                    if event.get('type') in ('progress', 'progress_state'):
                        progress = data
                        if event.get('type') == 'progress_state':
                            progress = data.get('nodes', {}).get('11', {})

                        '''
                        Ignore progress emitted by loading or decoding nodes.
                        忽略非採樣節點的進度，以免誤顯示百分比。
                        '''
                        if str(progress.get('node', progress.get('node_id'))) == '11':
                            value, maximum = progress.get('value'), progress.get('max')
                            if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:
                                pbar.total = int(maximum)
                                pbar.n = max(0, min(int(value), pbar.total))
                                pbar.set_description('採樣中', refresh=False)
            if elapsed - last_history_check >= 4:
                last_history_check = elapsed
                if not ready():
                    '''
                    Preserve recent server logs for actionable diagnostics.
                    保留最近服務紀錄供後續除錯。
                    '''
                    raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])
                item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

                '''
                Stop polling once the job has a final record.
                任務產生最終紀錄後停止輪詢。
                '''
                if item:
                    errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']
                    if errors:
                        raise RuntimeError(str(errors[-1][1])[:5000])
                    assert item.get('status', {}).get('status_str') != 'error', (
                        f'ComfyUI 工作失敗：{item.get("status", {})}'
                    )
                    break
            pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)
            pbar.refresh()

            '''
            Avoid tight polling if the WebSocket connection is unavailable.
            WebSocket 不可用時稍作等待，避免過密輪詢。
            '''
            if ws is None:
                time.sleep(1)
        pbar.n = pbar.total
        pbar.set_description('完成', refresh=False)
        pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)
        pbar.refresh()

    finally:
        '''
        Always free the live progress widgets and connection after this job.
        無論任務成功或失敗，都釋放進度列與 WebSocket 連線。
        '''
        pbar.close()
        if ws is not None:
            ws.close()
    files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]
    files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]
    assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'
    source_video = max(files, key=lambda path: path.stat().st_mtime)

    '''
    Save the user-visible output in 設定的工作 storage.
    將成果儲存至 設定的工作 目錄。
    '''
    OUTPUT_DIR = ROOT / 'minimax_h3_outputs'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{EXPECTED_VARIANT}_{QUANT}{source_video.suffix}'
    output = output_base

    '''
    Number duplicate filenames deterministically.
    依序為重複檔名編號。
    '''
    for index in range(2, 1000):
        if not output.exists():
            break
        output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')
    shutil.copy2(source_video, output)
    print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')
    LAST_OUTPUT = output
    print('✓ 第 11 節完成：影片已生成；請執行第 12 節預覽與下載。')


12. 影片預覽與下載


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片。'
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable file link for the generated video.
顯示本機影片的下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))

'''
Keep the preview cell independent of GPU vendor-specific monitoring tools.
預覽 Cell 不再呼叫 NVIDIA 專用的 nvidia-smi。
'''
print(f'✓ 第 12 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


13. 擷取最後影格（選用）


In [ ]:
'''
Count decoded video frames to identify the final frame.
計算影片影格數，找出最後一個影格。
'''
import json
from IPython.display import FileLink, display
from PIL import Image
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片，並確認第 10 節可正常預覽。'
assert shutil.which('ffprobe') and shutil.which('ffmpeg'), '找不到 ffprobe 或 ffmpeg；請先安裝 FFmpeg。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：TASK = 'i2v'，FIRST_IMAGE = '{LAST_FRAME_PATH}'")
print(f'✓ 第 13 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


14. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'fps', 'steps', 'ref_image_size', 'audio_shift', 'quant', 'video_audio']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xd5}ks\x13W\xb6\xe8w~E\xdf\xa9\x9a\x924\xb1\x15\xdb\x90d\xc2\x1d\xcf-\x06\x0c\xf1\x9d\x048\x182\xe7\\\x86R\xb5\xa5\x96\xdd\x83^\xa3\x96l<>\xae\xb2\t\xc6\x06c\xec$\xe0\xc4`\x9e\x01\xe2\x81\x80!\xbc\x8c\xb1\xcd\x87\xf3O\xeeQ\xeb\xf1\x89\xbfp\xd7c\xef\xdd\xbb[-c2s\xcf\xad;S\x15\xacV\xab{\xed\xb5\xd7\xfb\xb5#\x91\xc8\x8e\xdel!_,\x19\xa5A\xcb\xc8\xd8\xfdE\xb3h[\x8e\x91\xb3\xac\x94\x952\xd2\xf9"|a;\x86c%Kv>\x17\xdf\xe1^Xq\'\xefV\x97~\xaa>z^[\x9eq\xd7\xe7\xaa\xe7\xc6\x1bK\xe3\xb5+g\xdc\xa9\r\xf8\xe8\xae=\xf8\xcf\xf1\xd3;"\xf0d\x9b\x9f\xfc\x17\'\x9f\x93\x7f;\x83\xe5\x92\x9d\x91\x9f\xcae;%\xff.Z\xf2\xaf\xacY\x1a\x94\x7f\x0f[\xfdN>y\xd2*\xedH\x17\xf3Y\xa3\xf4\xd7T6n\x96KyC|\x8f\x17v\xd0\xcb\xfa\xac\x0c\xc0H\xcbHY\x8e]\x04\xe8K\xa6s\xd2\x18\xb6K\x83\xf9r\xc9\xc8\x0fY\xc5\xe1\xa2]\xb2s\x03\x86\x9dKY\x05\x0b\xfe\x93+\x19\x05xn\xa1\xe4\xc4\x8d\xbd\x83\xf9\xbcc\xd1\x03\x06\xac\x9cU4q\xc1\xfc\x0c;g\xf41\x02\x8c\xae\xf8\x8e\xc6\xc4j\xf5\xdb)D\xc1O\xb7j\x97nT\xa7\xe7a\xf1\x95\xd7\xaf\xdd\x99\xc5\xb7\xeb\x17*\xab\xb3\xf5{g\xdd\x95\x07\xee\xe4\x0b\xf7\xd1W|\x1dn0\x0e\xd3\x8b\x007\x86\xbb\xb4\\\xfb\xe9\'\xa3\xcb\xa8\xad\x8c\xf3\xc3\xde\xae_\x89\x94\xba\x86"mF\xc4\xe6\x7f\xd2\x994\xffQ\x84\x7fh\x81G\xf7\xf4\xfd\xd1\xe86\x062\xf9~3\xe3Dc\xf1\x01\xab\x14\x8d\xe0U\xbc\r\x7f\x1dcL\x1c\xb1\xcc\x14-\xc3!\x94\x08DD\x1cX\xaa5d\xe7\xcbNf\xc4\xb0NY\xc92~\x95\xcc\xe7\xd2\xf6@Y\xac6ie2\xf1\x1d\xf5G\xe3\xee\xdc\x02\xee\xeb\xc4*\xc3\xef\xbe\xfc\xd9\xbd\xf1\xb2~\xebBc\xe2"\xac\xa5\xbe\xfc\xd0}t\xc5\xd8\x0bw\xcb\xbd6\x1d\xc7\x82\xfd  \x01]!P&\xf6\x1e:\xb8\xbf\xf7@\x1f@;:\x16k3\xd2\x91\xfa\x83\x19wr\x9a\x1fl\x8c\xe2=\xf1r\xa1`\x15\xa3\xb11\x03\xe9i\xee\xab\xea\xe5U\xff\xbb"r\x89N>3d\x85\xaeR\xdf\\\xc7*\xe1\x86;@\xb7s\x0b\xee\xe6w\xfa\x9a\xe0\r\xb5\xd9\xe5\x1a\xc0@\xef\x91\x0ba\x84\x00\xa2u\x98\x8f\xe3\x87\x13\xfc\xee?ZV\xc1\xc8\xe7\xe0\xc5\xe5\xfe\xac\xed8\x88\xb7t&?\x0c\xb4m"\xd5\x99\xc9b\xdeq\x0c3\x93\x01\xf6)\x17\t,\x04\xe0\xea\xd5\xda\xf2#\xa6\x97\xea\xf2-\xe4\x94\xc9\'\xb5K\xcb\xee<\x90\xccx\xe5\xf5\x86\xfb\xf2nec\xa9\xfa|\x02\xd8\xaa:7_Y\xbb\xd38=W__\x91\x80}qh_\x8f\x00\x8b\x019&\xa8\x95)\xd8pJy|= \xbf\t+b_+\x1bo\xe0\x85\xb5\xab\x8f\xdcs\xb3\x8c\x03\r\xb9F\xfd\x87[>2\xa5w\x1c>r\xe8\x8b\xc3G\xe1\xb5\x8c\x96\xe3\x11~Y\xe4\x84\x1f\x04\xf5\xb6\xa4\x99\x1b2\x1d\x1d\xefw\xbf\x83w\x06\xf0^__u\xa7\xd6\xdc\xc7k\xee\x8a\xc2\xfb\x9fz\xf7\x1d\xfd\xac\xcd\xf8\xac\xa7\xf7\xc0g\xfa\x0b\x87\xedTi0r\xa2M]\x18\xb4\xec\x81\xc1&\x08p\x99\xedN\xc1J\xdai;i\xa4$A\x9b\xb9\x94\xe1\x98\xd9B\x06\xd9>\x99/\xe7J-@\xaa\xfd\xf8\r\x12\xdb\xf4T\xf5\xe2\xad\xea\xf2\x0f\xd5\x87w5\x9a\xe8\xeb\x01:\xd8\xd7\xd7f\xf4\x1d\xed9\xdc\xa7\x01\x07b1\x9fK9:xN\xc9*8-\xf1\xb3\x1f~NB\xd5B\xf9ef\x8c\x8c\x95\x1b(\r\x12\x9cx#H\xaa\x02\x08+xX\xc9\xb4A\x08\xc1\xa6m^\xab~{\x857\x8d~^_\x9e\xae=\xfa\x8e\t\xc9\xddxR\xbd\xb9\x0e\x90\x92\xe0\xb9\xc7\xdb)\xb0\xbb\xf1\xa4vn\x8ao\xa8]\x9c\x92K\xd9\xef\x83?M\xa0\n\xde\xb5\x1d;\xe7\x94\xcc\\\xd2\x8a\xc2]m@I\xa5\x18\x01\x96\xcb7\x7f\xd9\x9f\xcfg\xf8\xdbN\xe3w\xdd\x04\x18\xfc\xf3q\x07\x88"\xfc\xdb}3\xd9\xb89]\xfd~\xc5\xe8|\xbbq\xfd\xe3\x0e\xe4\xe7\xea\xe5g\x8cS\xc1\xc4{\n\x05\x90D\xcd\x9b\xe7X\xc8E\xf0d\x81\x8c\xb4\rH2\xb3V\xab\x9d[~\xe4>\x9c\x87\x9d\xe3uW\xef_r\xe7g\xbd\x9d\xeb\xd9\xd7f\x1c:v\xf4\xf0\xb1\xa3\x89\x83{\xbe\xe8\xf1m\x9e\x95\xd2w\x8e_\x97\xc0W\xc9\xfdSr\xc6\x07`\xd1J[E\x0bP\x01Z\xc8\x1c\x00m)uL\xd6>\x85t\x96\xcd\x83\x06\x92;\xa7C\xcb\x0c\xe8>\x9e\xadN\x9d\x03\xa9S\x1f\xff\xca]Z\x80]\x82\xddkL\xbcq\'gAy\x80<`\xf1P}\xf9\x12nV\xfb\xd6{\xa4\xefh\xa2\xf7\x8b=\x07z\xda\x8c\xcf\xf7\xc8\xbf\xf5\xbd\xb4\x8bN)A\x10\xe9\xab\xca\x98\xde\xd5 \xde\x83\xa2\xd3,\xa7\xec\xbc\xe1\xd8\x03Y\x13\xa4\x99\x9d.\xa9\x95Y)\xd6\x9c\x01\xed8P4\x0b\x83R\xb0T\x1f\xdeQ\x9b\xd2\xb8\xf1\xb4\xbe|\xde\xe8\xa3G\xf5\xe1\xa3X5\x02]T\xde<\xaa^z\xc5\xf4k\xfc)_<\x89\xc2S.s\xcf\xb1}\xbd\x87\x12}\x9f\xf5\xee\xd7e\x00\xc1\x95 \x88\xde\xbd\x88\x03\x07\x8e\xed7\xfeZ6s%\xfbo\x0c%2\x1c\xeeI\x06\x1f\xe8\xd8\xc0\xa1\xb9\xe4\x88\x91\x1c\xb4\x92(\x98\x03\xc0W/L\x01\x0f\xa1$\xa4\x075\xa6\xe6\xdc\x0b\x0b\xc0C\xb0#\xb8\x84\xd7wk\xb7\xd7*o\xae\xe1\x1e]\x9f\xa9\xde\xbf]\xbdq\x17\xe4w}\xea\x99\\\xc2\xbf\x1c\xdbsP\x07\x9e@\xf1\x08\xea/h\xa5\x94s\x0eh:0`\x00l\x1f\xa8H|\x8e\xd1o\x01\xc8\x96a\x96PL\x10\xde=\x9c\xc7w\x08\xd6?7\x0b\xca\xb3z\xe9Ru\xfa> \xb6zi\xa5:\xf7\x0c1\xaf\x01\x0c\x8cP[~\x12P\xd1\x0c\x1f\xa8\t\xfa#\xd1\xd7\xfb\xbfz\xfa\x12\x07z\xff\x80\x1aY\x7f\x10\x7f\x0f\x96\xc9(\xfd1&\xb8\xf6\xf3<X\x17\x8a\x03\xda\xf39\xd8\x86\x82Y\x04\xb8KV\xd1c\x053\x9dF\x8b\t@O\xe3w\xed\xfd\xa6#\xb6\xc8\x91\x06\x86q\xa4\xebK\xc3}<\x8d,A*X\x90\xc8\xc6\x93\xc6\x8d7\xee\xc5\x1b`a\xe1z\xee-\xb8\x8f7\x85\xa0#\xc6\x90\xeb9\xd2\xb3\x9f\x99\xc0\x93hlm\x00tL\xf1\x0e\xd8\x1a\xc7O\xc4|\x06b\xbe\x80X\x04\xc9\xebq\xf1\x90\x9d\xb2\xf2\x0e\xeb\n\x10q)3\x83\xea\x9d\x88N\x1a\x0e\xc0\xc2\xa8:\xc9(A\xb6%\xe1\n2\x87-\x08\xa6v\x1d\xae/{\xf7\xf5\x1cB\xb82@n\xd1 p\xfcB\x06.F\xf7\x13\xdd\xb7\xbc\x9f@Q\xf7\xd3\xb3\xf9\x17\xf0\x03\x14\xc4\xbe\x1f\xd0\xc3\xf9\'\xf0\x8b\xfd`\x85Y\xb1\xa0e\xe8\xad\xdd\xb1\xff\x06\xff)\x01mY\x03#\x06\xed&\x1b\xfd\x16\xf8\x00\xc3\xb4E\x05\xb0\xc7\x11\x0f\xf7\xabK\xe7\xe8\x82\x10cJ\x80\xad\x02Co\xd6\x1e.\xd4.\xdfm\xda\x1b\xa2\xafV\xfb\x93\xc0\xb7\xa3\xf5\n&\x7fr0\x12k\xb2\xe4\x01D\xe0\xdb"\xc0\xeb\xc0\xaa\x90k\x15\xa8\x12\xcad\xb9XDS\x0fIK\xda\xe6\xac-\x99Z\x90\x92{\xc4c\xea\xeb\xe7\xc0y\xf1C\xbag\xef\xd1\xde/{\x12G{\xfe\xf5hB\x98X:\xac%\xebT)\x81\xc2\x03\xc0T\xf7\x08@\xbf@\xa8\x19U\xe5\xacUD\xcd`\xe6\x06\xd0\xffHf\xc8\x14\xecg\te&Ke \xb8/\xec\x9c\xfd\x85y\xea\xb3\x9d$\x15I(\x82jM\x81^\xab\xae\\\xaaM\xfe\x18z\x03\xba\x08\xaf\xaf\xbb\x8f~p\xbf>O:t\xd5\x1d_\xaf\xad\x9cq\x97f\x1b\xe3?\x83\xe1\xcc"(\xc0\xe3\x1d\xf1\x0eR\xcb\xba8\x85\x8f\x9d\x1d\xf0\x05\xa0[\xbf\xcc\x9a\xba\xf2z\xa6\xba\xb0\xce\xbf\x03\xb2\xe6;\x8d\xca\xab\x99\xc6\xc2\xb7\x9e\xc6\xee\x01\xb1\x05r\t\x95\x01\xac\xee\xc0\xe1cF\x87\x91\xb5\xb2\xf9\xe2\x88\xd1_N\x01\xbe\x1cc\xd0\x04u\xd9oY`z\x81\x84\xb6\x91\xe7\xf3Lr\xe5\\\x0e\xc5\x01\xc07d\'a\xd1 G\xeb\xf7g\xeb\xf7\xdf\x80]\xc0n\x85xb}\xf9\xfb\xea\xd9\x17\x8d\x07\x97\x1a7\xe7\xc1\xd2A\xdf\x83\xf4\xbe;\xfdXl\xec\xd2,\x08\xea\xc0\x92\x03\x1e\xc7g;\x13bg\xd9\x84\x8f\xc4\x8c\xeen\x03\xae\xee\xeb\xe9\xeb=\xd2\xb3O\\\x06d\xf0K\xd5\xbb\xea\x8f\xceW\xaf>\x03Q\x04\xdeIcj\xb6\xba\xf0\x98\x1d\x14\xf4\xd8v\xe1v\xb8\x97o\xb83\x97\x8d\xbd\xf9lz\xe4X\xaf\x87\x9c\xc3\xe0]!!j\xea\x11\xccM;\xc3T:\x08\xbee\xce/\xeb\x87\xc1PN\xe5\x87s\x19\x90\xa8\xc2\xe2)Z\x03\xa8\x9e\x80v\xe2F\xcf\xa9\x02\xb2\x81i\x1c\xb1\xd2]_\xee\x01\xaa\xca\xe5mX*3\xe9\xf0\xa0\x95\xf3\xb3r;\xb1\x94pX\x1d\xa5\x16\x81s\xc1\xa8\x9d\xbb\xea\xd3\x0b\x9b\x17`\x81\xbc\xf3\xe8\x81=\xba\x00\n\xa5\xb2:\x03\xfc\x81\x9b/\x96f\xb0\x8e\xab/\xbfv\xcf\x9e\'\xbf\x95D\x00\xb0\x98\x14\xde\xd5\xc5\xd3Up%H\x1cH0\xdd\x8b\xaf\xdd\xc5\xfb\xfcS\xb9E=\xffz\xb8g\xefQ@\xfa\x97{\x8e\xf4\xb2vD!\xd05d&\n@\x17`\x86\x19v\xda`\xee\xebf\x97\xd7\xb0@n\xa1\x1b\xac\xdd$\x85\xd8_\xcb\xe0\xde\x83\xe0+1\xf3\xb1jO\x9bY\x1b\xf0"\ri\x1f\xa6\x81\x04\xb3\x8aWYP\xa0\x99\xb5x\x9ai\x90\x81\x05;\xb2q\xeb\xba;w>\xa0\xf2\xd92k\\X\xa9\xdf\xbfS\xfd\x99\x96\xba\xffs\\i\xf5\'\xb8u\xf6\x9dd\x88\xcb\xfa<\xb1\xf7\xb3C\xbd{{\x98\n\xa3A|\xb4\xb1\xaee{\xba%!\xd3\x83\xde\xf1\x846#\xba\xc3\x80\xff\xa5#\xcc*,\x0f\r\xa1\xc2Q\xc1n\x9cu\x1f_\xa9.\xdd\xd77\xbd:\xbd\xc0<\x05\\\x00\xf2\x04\x96\xa4\xbb\xe6*V\x81\xfaNr\x01\xd1}\xcc\'\x16\x10\xb9\xc3\xc2\x9eSF;\xf8\xc1\xc3\xe6\x88\xc3\xf8\x07\xd3F\x17\xd9\xfa\x1eIy\xa0,B\x83\ry|\xa7\xb6\x92\x80\x81\xe3\xb7\xbc\xc4\x16\xec\xeb\x95\xa6~7\xe0!\x0bB5k\x9eJ\x0c\xeeL\x8c\x06\xb16\xd6.\x10\x13\x1f\x18(\xa7Q\x11I\xb3N\xbcZ\xd8\xb3D\x19o\xd7\xe7\xd8t\x02\x1b\x85\xdf\xab\x84D\xfd\xe5\xd7\nI\x9a\x7f3\x94\xb7\xc1\xa2\x18\xcc\x0f\xa3\xdc3\xd1\xb6\xc8 \xbf:\xe5\x0c85\x18\xce2s\x86e\x16AJ\x16\r\xa7\x9cLZ\x8e\x93.gPR\x82\xf9\x04R\x00~\x936\xed\x0c\xe0\x16\x15\x1bR\xa1\x1etr\xef<\xa9^\xfe\x8eY\xb9\xba4\x0e\x7f\x00T\x80\x96\xcak\x10\xaaw\x1a\xb7Vjw\xd6\xea\xd3\xe7\xd9T\x91("\xcf\x81\xdd!`\xc2\x83`\xe70\xac{Q\xe5\x15\xb3\xactO\x81\x0cB\x903\xf9$\xa8-!\xb0I\xa6\x808\xcbHI.w\x0c\xe9\x04lQ\x92\x8a\xa8\xa1\x00\xc4\xbf\xdf\xa8^\\\xae?\x10D\x05\x00U\x1f\xfe\xe0\xae\xae6&f*\x1bK\x81\xdd*\x82M2\x12\x05\xc2\x8dH\xb1\x03B\xb8\xb2\xfa\xb0\xba\xf0\x92\xf1\x0b\xcf\x0f\x95\xc3\x1e\xa2\xbf43v\n\xac\x82\xa0w\x82\x82\xa1M\x84G\xda\xbc0\x00y\xe8m\xc4k\x85\xbc\x03~\rh,\x190\x00]L\xfaT\x8f\xd3\xfc\xe7\xf8\x84\x0c\x8aL\xe8\xd1\x01\x10\x15\x1c5\x007\xd7\x9d\xbf\x07\xa2\xb1zy:\xb0:\x92i`kG\x03\xe1\xbdLH\x94\x8f\xd9\x9f#.q\xb0s\xecB\x94/q\xd4\xe1\xf7\xa0\xa5\xf0\x13\x86G\xe3\xb6\x93\x06\xba.YQ\x11\x9a\x107\xf2\x07\xbcU\n\x02\x0c\xb2\xf1>\xb5Z\x12?\x1d\xe5\xbe\xfc9\x87\x11`\xcf\xe0\x87o\xd7\xaf\xca\xcb\xac/j\xa7\xd7\xe0\x1b\xe9\xc8\xc7\xfc2\xd9\xc3\xa7\x9d\xb5r\x18\x1a\xc3@\x98=\x90c;`gW\x9b\xe7 \x00\x91\x17K\xe8O\xdbd\x8c\x83\xa3]\xa6]\xcb\xd8Y\x1b\xa34+?5\x1e|\x8f\x91e\xf9\xc2\xca\xea\xa5\xdaO\xf7\xdc\xf9ix\x0e\xc5\x07\xc7g=\xa7a\xee\x9e{g\xa1\xb18\xefN\xbf\xa8\xff\xf8C\xf5\xfa\xbc\xbbv\x0f\xc4Z\xe3\xf2Kx\x10\xd8W\x81}\xa1h\x93B\xa9\x08:\xc9\x8f\xfc\xe5\xaf\xf15 B|\xb7\xa8\x8b\n\xbf|3\xa2O\xdc\x12@\x13@\x8da\x10\x1dd\x1f\xe6\xf2\xe5\x9c\xb4\xc9\xffZ\xb6\x1c4\xacT\xf4\xaa\\@\xac\x01\xa6>\xdb\xd9\x9e\x84\xed\x82\xab\xfd B:?\xc9}\xf0\x11\xfbU2\xa6U\xd9\xbc\x86\x16\xe4g;\xf1-\xe2\x86\xfa\xbd9\x97b\x0b\xee\xe3)\xe1\xd5\x12\xb9\xba\xf3_WV\xcfc\x84\xf7\xf23\x00\x13\x10Z}zYE\x92T\xc0\xe1\x08\x88P\xf4H@xF?j\x83g\x1a\xbfa\xdaKZv&*\t\x0f.b\xbc\xa7\xdd\xf8(f|\x087\xc5\x8c\x0f\xe0O\x15=)HK\x1eVi\x153#\xc8\x7f\x92\xcf\xd1\xf6\x05\x7f\x9d\xd7A\xbb\xdeF\xa1&\xa2\x8c~\x1b\x0c}\xb0+\xe1\x82\x05.\xceI\x8dC\x1b\x13\xcf\xddG\xd3\xea1l!\xd7\x96n \xc1\x02\x8a\xe5B`\x8d@\x0f:y\x88h\xe7A\xf9H\x15\xf6\xe3[\x03$"\x10\x00f\xf3\xce\x8f;:(^}o\xa2\xfa\xe4\xb41\xca\xdf\x8c1\xe4\x0eJ\xa9\x17\x93\xee\xd4\x9a\xb4\xdfi\x17\x08(~.\xfd\x1e\xf8\x089q\xf5Q\xed\xc6C\xc9h\x9e\x04;\xc0B\xcbB\x89\x0f\xac0\x08\xe4\xd2\x0e\x18\xe0P\x98g\xef\xb5w\xa2\x14V\x84"\x9d2\xb1.\xf8\x96\r2\xf7\xf5\x1a8\xa5 1\x91\x16\xc4\x93\x1aW\x96A,s\xac,\x90\x9fq\xacd\x11Lw\x7f\xc4\x8b\x02\xdc\xf8\xf6\x02\xba_\xa0\x91\xc2\xf2(\xe4\xfaHl\x8a\x98/z3sJIU\x9f\xac\x01h\xa0\x98\xdc\xf1\x19~9\xdb *K\xe2\xdb\rp\xfa\xdd\x1fgx9\xca\x93<v0\x81Q<\xa0D\x01h\x1c\xbc\xacT\xbf\x056B\xb4\xeb7\xbf\xd9\xd9\x15C\xe3\x91o\xe9F$\x90\xe5h\xe7JQ\xbc\x16\x10P\xa0\x85U\xe0\x85\x96\'B-\xfde;\x93\x92K\xb2s\xd2;\x96\xe6\x0c`ziY\xa0\x954\x9b\x1e\xac\x07\xed\x87L69-T\x07-\x93]\xb4\xa0[\x86\xd4\xa4\x16\xf4;\x83\xc0\x07\xe9O\x1f\xfd\x8e\x98Q\x9fzj\xec\xea\xfat\xd7\xa7\x1f\x7f\xd2\xf5\xe9G\xa8\xe3\xa7\x91\x80\x81aa\x8d\x1e\xe5\xf4\x81y\xa1;\x99\xb4(\x07\x84\x06\xfa\xa2)\xb01p\xef\x92\xc0R\xfd(a@\x17\xa6\xcaI\xb8#\x03\xc4V\x04V"+\xc1]\xb9\xd7\xb8rUd\x07d@\x15\xdf\xb8\xf0\xaa\xb2\xf9\x06\x9c@\xb0-@\x05\xd7\xe66\x05\x15q\xd2\xeb\xf9|\xf5\xba\xd2\xe6\x85"b<\x1d\xd1-\x14\x02\x06\x03Hr\xc9c\xd2\xc5\xff\x03\xa2\x1b`,\xe7l\xa0e\xc5\xca"\xe0[\x00\xdf\xc0>\xc5&\x12\xc9\x0e\xf4z\xbc(p\xf5\xe6+\x8c\xab\x92DcK\x8dw\x06h\nm\xb1;h$\xaaG\x8a`\xf8\xb9\xd9\xda\x0be\'\x8a@/\xb0P\x16\xa8\xea0\xc8\xb4\xa8\x16!\x8e\xc5\xf1\x0b\xe9\xe1\x9f\x04{\xc8\x04\xcf\x01\xd1\xa8g\x96\xd0\xd7c\xa6\x94\xb9\x0e\x10Z(\xe6\x92\x83\x18\x03@\xde\x9c}\n(eb\xae]\x05\xa9\x8b\x86Z\xe5\xd5\r\xd4\x0c\x1a\x87T\xde\\\xab]^\xe48\x12\xac\xa6zK\x99\x10\xf0J\nF\x1bh\xc6\x8ej@\x8f\x85X\xb2\ta\xc9&F1\xa9\x1a\xc7\xff\xec\x02\x17b\xd0:u|\xf7oO\xc8\xe0\xdd^\xb0\xb6P\xca\xa0F\xf1lu\xd4o)\xab$E\xf5\xbf\x0c[\xb9\x9d\xed_~\xae\xa2/\xb4\x1168\xf7\x142-\xe4m\xd28\xec\xf5)Y\xc7{\xa0\xccwNH\xb8\x9b\xd7\xd9K\x82\x95\xd5\xa7\xee\x83\x85X\xdf\\\xae?\xfc\xde{G\xed\xe5r\xed\xf6\xba\xbb\xa8\x02\xdd\x14R\x86%\x8f\xb2z\xed\x8c\xec6F#\xc9\x0cpP\xa24R\xb0\xe0cdo\xc6.\x80\xf9\xf2\x17\x0cEZE\xb2\xaar\x80\x1b\x07o\xddA?\xa3\x9f&\xe16\x8e\xe6\xc3o\xfe\x8a/\x1c\xca$v\xf5\'\xd2\x85\xdf&\x1c\xb0la\x93\x1c3\r\xeb\xce9\xf9\xa2CIT\xf1\x02\xcc0G\xda\xbc\'\x01\xdb\xfc\x85\xf3\xbf\xf8m6;\xb8\xb3}W\x7f\xbb\x04\xa3}hg\xbc\xb3=\x9b)\xf8\x1f\xe7\xfd<e\xa1\x01\r?\xa5\x00\xd2\x81\xc3\xc70\xe2\x85\xcao\xb7\x11\x8c>\x8d\x8d\xf1\xef"]!\xeb>\x96\xb3J\xbcf\x0cM\xfb\xd7\x1d\x01\x0f\xb9$W+= \xf5\xb4\x9d!O\x0b\x896\x05P\xa9\x16@\x1e6\\:\x0e`\xb5\x19\x1d\'\xe06\n\xc6s \x13\xbe\xe8\xec\xa2\xb8\x12_\xe4\xd0\xe3n=\xf8\xa4\x00\xd9\x15\x02\xc8\x97{z\xc2v22dZj\xfb4/\x8e\xe3\x9b\xf8e\xba\xd0\xf9\xb1\x0f\xe7\xea5\x1f\xfd\xe3\xaf\xe1\x9c\x03\xbffgW\xf8k>\xde\n\xad\xbd\x18\x8e9\x9a\xff\x92P\xd4\x02\xb1H\xa2\x84\xd7N\x89Wx!]\xd8%/\x88\x84\xebn\xe1\x18hd\xc5\x99\xd1\xdd\x86\xc8\x9c\xca\xc4\xe8na\x88\xa2\x97A\xc9E\xb8",\x16\t\xf7\'!p\xff\xc1t\xec\xe4\x01\x10\x1bM\x1c\x15B\x06;%t\x98\xfe\xb4\x911@\x02\xd27\x1f\xd37\xeaM\xbf\ry\xd3\x1f\xfb\xd0\x13\xb3\x8a\x1c\xe7\r\xec\x86\xc3\xdf\xa9\x1d\x01k(\x91\x05\x87\x19\xc3b\x05\x0f\xf3\x9f\xb6ZA\x1f\xc8\xa8T9\xf3^\x8bp\xd4o\xe0)\x8e\x8d\x00\xe0\xaf9\x93\xbb\x9b\xdd\xaf6dc\n\xc0!\xb9\xc7;\x14$\x9d\x1d!\xa0\x1c!\xdd{\x90n\xf7\xaf\x8f\x1e\x91\xa0\xd4\xe3ne\nx\x0f\x0b\x13w\x02]{\xcbN)\x9f\xdd\x93\x1a\xc2$l\xaa\xd5\xf2$\x88\xc7\x110\xb1\xbe\x01\xdeU\xbc\xf8\t_\xf3\xee\x17\xf8\xa6/\x7f\xab\x10\x82\x12\xc1\xa1k\x9f\xcakh1\xe4d&Q\xeet\xa7\xb7\xd3\x9d]\xe1<\xb7\xcfJr\xf4\xbcy\x97\xf9\r\x9d-h\xdf{\xf2\xce\xad\x9e\xbcG$:\xb6\xff\xf8\x8f\x02\x8f\x0f\x93I\xac(\xb7\xe4]\x91a\xa2w(\xb1(e\xdfq\x84Z\\K\x13\x15\x81\x97\xe4\xbd\xf2\xa3\xd0m\x1e\xda\xfa\x85R\xd8\x1eG\x90\xe5\xb3\x85U\x94`\x9b\t\xbe\x95V\x83\xb6\xc7`\xe1\x82\xe3\xa6\x14\x1b\x821\xe6\xd5\'p\xdc6\x82\x11c\xcc\xe3\x95\xd0\xa7\xf4\x98\x9a\xbc4\xb2\x10\xccR\t\r!\x82\xe2C\xce\x14\xc3&8\xf1\x1d\x14\t\xc6\x8c\xee\xb2\xb0bn_\xab\xbc~\xc1V\x01\x9b\x01\x8d\xf1\x1f\xaa\x17\xef\x8a\x8a\x84\xb9\xf3"9\x0c\xbfV\x9eH \xfa\xbb\x9b\xd58~\x85\xff\xeeE\xbb\x03l\x13\xb3$\xb3-\xfd\x18\x07Oky\xac\x82\r\xf6/\x08\x0b#ka\x1a\x82n3)\x14!\xdcL\x11V\xc0\xc7\xb1\x9d\xae\'\xe0\xdd\xf5\xe7\xe0\x06rt\x01\x9c\x10\xb0\xcc\x18|\x0e( \x94:8\xcd\xf5\x12^*\xb2\xcd\x88\x96\xca@|m\x94\xc9\x8bq\\\x86\xec~\xd8(\xed\xbe\x18^\xfa\x14v\x90\x92\xa0\x04\x04\x06\xf1\xee\\1>5\x00\x1a6\x15\xdd\xb9\x95\xca\xeb\xbb\xe8\xe4\xcd\xcdW6\xaf2\x021\xaa!\xf3\x8d\x11?\x9eB\xb2\xcbZ\xa6\x0fL\x1e\xdcPN\x80:\x14y\x1c\x11\x08\x99\xc3t2\xe7\x92kgnU/O{\x99\xce\xa5\x05\xae\xd3iL\xac6nN\xb6@E \xddG\x01/\xce\xe9Qr\xefT$\xc6+M\x04n\x84\xa5\xd5\xbf\xda\xc0\xf8\x08\xc7\xe7am\x18oh^Y\xcf\xa9R\xd1\x14\xf6i\xa7H&\x1f\xe6-\x17\xe4\x00\x0b:U\xc8\xd8I\xbb\x04\xder\xd9\xf1*\xa1X\x81\xf2BE\xe2\x99c_Fe\xf5a\xc0\xf5Q\xcf\x04\xb3\xb4\xbex\xe3\xed\xfa4X\xb0F\xa7\xd1X\x98\x01\xc7\xf4\xed\xfa\xb9\xe0\xfa\x05\xd9%$\x0c`\xb7\xa2\x1f\xc4\x1fc\x94\xa4\x14\xd4\n\xd0\x14\xadx\xda\xc6\xf4r&Z\x8c\xfcN\xbc\xea\xcf\xce\x07\xd1?\xa7>\x88\xfd\x1eP%T\xbc\x91\xce\x98\x03N7\xdc\xde{\xe0\xe0\xa1#={\xf7\xf4\xf5\xc4\xc6\x82[\x9d.cqQ\x9e\xcb\xd00O$\xea\xc0\xe4\x8eS\xd09g\x98\xfd\x0e\xfa+$\xad\x04\x12f\xeeJ\x0c\xa0\x13\x85\xd1\xa0Y\xf7\xe1\xf7X\x95\x08\xbb\xee/Hi\x8cO\x80\xfb\x14\xa0\tf\xf0\x16\xa4\x80\xab\xa3\xdc\xa3Xw\x08\xe5\xfb\xb1\x12@\xa1Jap\x1a\x03\x89F\x14\xa1\xb9\xeb\x97a\x97*kgux\r\x89G\xe3\xe0\xef\xd1\xd5t\x88\xec\xa3\xc1\x87\x8e\xbd]\xbf\xcay\x04\x91\xc7\x1e\r\x005\x86\\Gt\x87\xef\x8d\xf9\x91\xad\x05\x98m\xa03x\xa8a\x0e\x98\xc8\xfd\xa2lv\xc8\xf2E\xb5\xd81,\xe4\x1d\x07\xe3u\x8c\xf5\xda\xd5[\xc0\xce@\xed\xb2\xde\xe4\x07\x0c)pH[\xf3W\x85\xdc\xf9\xfbw\xf5\x87\xe7\xaa\x0f\xef\x84s\x1d\xf9a\x89\xe4`\x1e\xdc\x8a\xa8gq\x1e\x91\xbc\xaeY\x9dM\xb9v?\x07\xc6\x82TU\x16\x15k9\xd4\x19\x19\xe9\x91\xeb\xe1\\\xbd\xa2\xce\xab\xf3#1B\xd9\xb1\xfa\xf4\x14\x17o\x88\xfc;\xb9\xbeX\xf4\'\xc3\xb2\xaa\xc8\xce\x0b\xf4\xe80\x90\x03\x88\xe6\xc5\t\xe4\xa6V\xd6u\xd8Z\x7f\xa1\x85\xad,}\xcd:\xf0;\x7f>\x03\xfc\x17\x18\xde\xda\xd3\x02\xfb\xb1;\xb0\x1fcMl\xce\ta\x0eB\x08\xaa\x96\x92-_D\xef\x9cd\x9d\xa84\x10\xb0:M2\xb2d\xcaM\xaal^\x93\xdc\xaf\x0b\xbcNL\t\x88\xbf\xbb\x8c\xc6\xcd\xb3\xee\xda\\m\xf15\xe6\xe2W\xe6\x805tm\x19\xdc1\xe4g^\x13FHN\xb5\x89\x0fX7\x82\xa0ZT\x1d\x01\xec\xa3\xf3\xdbn\xcdQ\x17\x8f\xc1\xff\xf5\x95\xf2\x05\x19\x93\xd329`\x1fP\xa9\x17W\xe0\xf9\xd4Z*\x8f\xe5\xea\xf9\x12\xa7\xad\xe2\xeaI\xaa\x8eK\x87\\I\x8e\xea\xe2i\x8e\xd9\xb9\x13K\xd5\x87\xb79\x14#\xd7\x15\x04\xca\xc9\x97\x8bIK\x86\x8a\xbc\xc5\xc5\xc2\xd7\xa0Y+\xa61\x84\xa2Cd\xd2\xd0X\xa3j\x1aJ\xfd\xc3^\xa1-\x95\xf2@\x16\xf94\r\xde\xfa\xcb\x15w\xf3k\xce,a"\x81\xe2C\xbc\x82V\xc0\n1\xcc0\xc7m\'\x81o\x8dR\xbd5\x19j\xe76\x11\t\xd3\x8f1\x8f6\xaa\xed\x9a\xf1\x81\xd1IB\xd0_\x98\x88\x12\x15\x9f4\x16\t_\xec\x9ed\xd2*\x94\xa4\xb8\xe0\x12\x04\xb67\x1d\x8e\xf9\x83\xd3\x92\'\xfa\x14\x12\x0e\x88\x13\xddp\xf2\x8eE\xf9\x8b|\x16Z[X\x8d,D\xa1w\x1b\nN\x8a\x1b\xe1v\xae\xae\xd6\xef\xcd\x08\x8b\x89\x92\xc0\xdb\xc3\x84SN\x83\x81\x1c\xcf\xe4\x87\xb1\xc0\x9cM\x94x\x01\xbcV`\xd7\xf8_\n\xf2_\x8b\xff\x18\xb6\xfa\x0b\xf4G\x7f\xb6\x10\x89\x05K\xe3t\x14\xc9:\x01\x89(\xf1\xa2\x96\xf8\xca -\x940C\x86I\xd6\xa4\x84\x8f\xf5F\xef>\x87\xb8\xa9\xd9\xae\xd5\xa8\x84\xe2\x87\x81\x8dr\xa7\xcf6&g\xd9\xfe\xa6\x02k*t"]\x02\x0fm\x85!|g\x02\xc8\xb3\x1b\xcb\xaa\xa2\x9d\x1f\x03\rh\x14\xd1\x8a\xbc\xf3\x85\x11"e,\xb5\x07\xfb\xc3\xdb\xd7B\xd1\x1e\xc2\xa5\x91\x1c\x86\x05\x16\xc1\x18\xc5\xda L\x1d\xc8P/\x158z\x8bq\x1fO\xf9\xe8\xfd\xceT\xfd\x87\r\xa0NO\x1f\xfe8\x81\x19\x8e\xf5UXO\xfd\xe9Tua\xd1\xbd\xb3\xc9^\x05+R\x15\xf5\xf5\n\x7f\xc3\xa0NY\x88n\x96&\xdd\xc6\xdeC_\xec\xff7\xe3C\xa12"\xf0W\x1a\xfd\x8eD\x13C\xb4\x08\xa7\x8e\x86\x12U\xab=\x17QWJ-\x10=\x07\x11\x04\xe2\x8d\x1bn<\xc4\x80\x91\xa6\x96\xab\xa4\x96\x0b\x1bMa\xd6m\xac2\x0e\xce\x1c\x18~\xf1\xecIxO\x94?8\xddG\x8bepMHT&\xf2\'\xe9c\x8bm>\x0cDg\x15E\x8f\x85u\n\x8do\x06\xbe\x7f\xa4d9^\x1e\x17h\x80\x92\nh\x0e\xc0\x1e\xe7J\x18 O\x19`\xae\xe7\x1c\x14\x04\xdafs\xa8[\x10,\xa5z\xdc\xc9\x1f\xddG\xaf8G\xd7\xb85\xeb\xdeY\xe0\xea`\xbf\x9ei\x92\xc7\xd4D\x14O\x02!vEy\'\xda\xf4\xa5\xb7$\xdc\\\x8e+\xaa\xb8\xca\xb3\x98\nH"\xaf\xddI\xf1\x9f\x8f\xed\x1a\xe3\x8b\xc8v\x0c?\'\x83\x96\x17\xabk\x0b\xdac\x98\xe5Z\x01\xce\xa6\x8d\xe0\xba0\xf3F=(\x10\xd0\x90Q\x17}\x7f\x91\x93\xa4\xb9\xd0$e\xc0\xcf\x1f(bO\n5Oa\xe6\x02LENQ\xfc\xcd*\xe6\xd9:h\x83\r\xcb\xd8\'y\x8b\x95! \xfd\x080\x19\xe2\xcd\x0fdN\xac=\xbb\r\xd68\xfaF\x1d\xca7\xba\x809\xf00\xcb\xc2\xe0\xaa\xf6\xad\x91\x82\xf6\xdeq\xb9\xe4\x13\xc7\xd3Z\x11o\xdc\xb3\x99t\xfe\x1c#\x03Qb\x13\x8d\xb6@5\x88\'D\xb3V\xca69U\xce\x06,\xbb\x05\x98\xdddx\x0b\x193i\r\xe63\xa8\xaf\xe2;\xd8\xfa\x9648!+h&\xd8\xed\xe6X\x81\x7f\xb1,\xc3\xd8il\x1d\xd3\xd0\x94RX0\x80 \x93_p\x011}\xb1\xd3\xf7\x05W\n\xf3\x17"|\xa0\x07\x0e\x14\xd4;\x8d\xea\xa3\xe7\nt\xfa\x14(V\x0e\xf7\xe0\xfb\xc0\xac4<\x08\xc4+\xeaSO\xdd\xc7_7\x96\xc6\xeb\xf7&0\x9b\xeb\xd7>\xe0\xb6c\xed\xd1\xa3\xe7zu4x\\\x02as+\x95\x8d%\xe4\x9f\xc53\xee\xf9\x9b\xca{\x8cl\x85\x90\x0f\x9a0\xf1A\x18\n:\xbbd\x08\x85w\xe0\xd9\xcd\xea\xb5yw~\xba\xbe<-\xb0\xd2\xd9e\xb8\xe3^f\x8d_\x8a\xe60\xb1\x9f R`\xad\x00\x00\xf0\xd8/E\xbc-\x00\x88g\xcc\x13%\xef\x11\x11\xbf p\x1f\x18\xd1\xe0\n\x80"\xf4\xdam\xcaSw\x80c/m\xe8\x8c\xd9oe\xda\x98N\xc9\xc6G\xa3\xdd.YY\'\x1a\xdb\xed)m\x15j8N\xa1\x06\xe1O7\x07\x18~\xedx\xb1\x05\xe3\xd7\xf2\xe9\xd2\x8b\xc1\xe8B\xecD\xd0X\xf2\xbcw\xfc\x0fC\xa2\x1e\xefy\xe8\xe9\xc8\xefF\xe9yc\xe0v\x8b`\t\x17A\x88\x80\xca\xeay\xf7\xf4S`\x93\xc6\x14\x99F\xe2\x97ca!+\xbd\x9d@\x96\xe2\x97\x1d\n=r\x00O\x9a\x02\xf4\x1d\tjM\xfdX\x18\x18\x02+\xa9\x00\x86u\xf2$\x89\x0bvs8=\xa9j\xca(\xa8\xc7\x84\xc92\x9aM$E\xb0`\xc6\x01K{e\xfet\'\xd2\xef\xeb\x0b\xa1\xee\x0e;:\xaa\xca\xb1\xc9\xcd\x11{\xeem[\xbeh\x0f\xd8\xd8w \xdc\x08\xf9\xd3Xp\x0b\xe4\x8d\x9e\xed\xce\xcdP\xf2r\xa8!\x9b-\xec"s5\x9b\x1f\x92\xf6k\x96\xedV\x9d%=\xf7gZ\x15P\xce\xca:\xaf+\xa3\xf2\x15c\xefc6\r\xeeL\xa8\x96\x860k\xa9\xb3\x0b\xcc\xa5\x16\xc0\x8f\xfd\xa3\x96K\xa8E _\x16\xb0\tZ\x18\xbd]\x1dh\xf5\x92\xc1\xf7\x1b\xa3\xcb\xbb\r\xde[r\xf4\xfb:\xc3\xef\xdb\x96N\x0f\tQP\xe8>\\\xa5\xfb\x1f-\x01\t{\xf6\x01\xabD\x8f\x062/\xe4s\x88\xa7`rS\xe6\x0c45\xd9\xf4\x86P\xdd\xcb\x9c\x18\xd7\xf6V\xd7\xba\x12(R\xbb\xf2i~\xf9\xb6\xdb\'\'\xdf\xf5&\xd1\xe3\x12\x0f\\h\xf5\xd6\xce\x13~1B]I\xc1\xe6\x1d\x03\x83?\x98\x0b(\xe6\xcb\x03\\\xfc\x9d\xa4\xbe*\xfa\xd2\xf3=Ce\x05\xeb.]Vx\xde\xd5\xc4\xcf\xf0\xad\x88\x85\xaa\x92\x8d\xf7\x16\x11BS\xfcW\x89\x88a\x93EC\xb6\xb0\x93\xfeM\x83\xcdC\x7f\xe4\x07\xd8\xe7\xcd\xee2u\x91\xc1\x08\xf8\xbf(2\xc4\x06\xff\xff\'2v)\x91\xf1\x9eb ,W)3\x86\xef\x94\x03\xa1\xbc\xa3qM\x08\xbf4\xd9\xc6{8\x87g\x1aCV\x91\x03i\x1c\x99\x12\xad#\xa2\xd6\x94?\xb6\xfb\xd2\x80\xecH\xc4w\x80\xc3.\xc2T2\x02\xcf\x05\xdf\xc2\x10\xbcx\x17\x9dwx\x84\xf8L\xb6\x1e\xfb\x0c\xd2.\x96\xdd\x98z\x98\xb0\xcdP\xa0\xd2\x02\x13\x92c\x8e\x13\x12\xa2\xa2l\xda\xd7\x04\x1b\xe9\xfc\x98J\xa8\xa9\xed\x95J1#\xc26\x8b\xaa\xba\xea\xf7\xfe\x81\xd7Y\x8b\xf7\x7fB\xa5\xdaf\xf3\xed\x99\x90\x9b\x7f\xdb|\xf3\x89@\x1e\xf3\x10Vr\x8a<\xaah\x1d.ra\xa2\xde+\x866OV\t\'\xccW\x00VA\xfchMml\x88\xabtMP\xfch\xae\x07>\xa8E\xacU\xd5D\xe6\xbc\x92\x7f\x11\xc0\xc3@\xa5l\xb0S\xfd\xb5Y_\xb0N\xb4\xcePw\xb3\xea\xb4\x13\xbbN2\x11n\x00cp\xeb`\xe5{FV[G\x87M\xcd\x86$\xe8\xc1\x89\xa7\xe1\x0c\xb9\x01_L%\x10\x06\xc6\x180\x10\xb1l\xa8\xc5a\x10\xb3O9$\xfc\x0b\xe2\xab*\xb6\xfa~!T\n\xa8q\xdd)\x86\x85D\xbf\x9c\xd4F\xbc#*V\xe4\x0f\x9d\xb5\x08\x9a\t\'\xfd\xea\xa3\xc6\x853\xbf< 6*x\xf2\x9f\x1e\x04+\x99\xc5\x01\nJPw?y\xdd\x9c1k\n\x82\xdd[`J\xe7h\x8bX\xae?\x16\xf8_\x1d\n\x0b4\xff+G\x84\xea>\x89\x85JM\x11/\x8euq\xdcH\x8fxQ\x03\'G\xbc\xfe\xc9\xb1.\x957\xf2\x82\x8e\xd4fX\x94\xb9\xa3=\x87{e\xf7~3Ai\xd1cp\xd8\xe1o\xba]\xef\xd5\xff\x7f\x1a\xdd\x92\x81<\xca\xbdr\xf3\xa4R`\xa0zt\xbd\xe5_\xdd\xcb\x9f\x85\xd7G\xcbl\x8c\xff \x94UPMm;L\xe5)\xabp}{\xc4j/\xb2\x83\x9b\xb1\x86L\xf0\xa5S\xc5|\x01{>\xb1\xc7?ib\xb2\x95\xbb\x17\xb1X\xc4\xc1\xc8\x0b\xd5:k\xfd\xf4Ys\x84I\xcb\x12=a`\xe4\x16K\xe5\x02U\xa4\xf3\x98\x18l\x8e\xa3\x06)\x16\xc9\x95\xd5\x99\xea\x0cui.<Ry|n\xcfr7/p{\xabhs\x9b\x16-\x11\x8d\xf1\x9b\xd4\x0b8\x8b\xc9\xdaG\xaf\xa4\xa6>\x08:$\xd1{p?\xb6\x99\x8b\xa6\x03\x87\x9a\x12A6\xfcaO_\xcf\xd8\x87\xf9~,8I\xd8\xb94\x9a5%;k\x01+t\x7f\xda\x11\x8b\xe3\xd0\xa7h,\xa4?\x0b\x17\x88S\x1a\x0e\x1f3:\rNa\xfb\x92\xe9\xa2\x8e]\xd6\x1e\xb3\x95\xce\xed-\xb5\xcb/PH\xcb\x8aa\xd1\xf8q\xe5\x0c\'\xc9\xb9\xaaG<\xd8\xdfC\xe0\xcf\x977\x97\x0f\x8b\x02\xdd6U\xa0\x1b\x06\xb8\x84\x88#\xdb\x16\x1b1^;\x93\xe3\x94\xb3\xc8\xfer\x87\r\xd8@\x13~l\xa2*\x12M\x90\x05\x14$X<-\xf2\xfb\x0bS\xee\xc3\xef\xd4z\x986Y\x97\xaa.\twb\n\xab\xfeE;\xc98\x96\xbc/\x9ds\xafbW\x85X\xf6\xea$l\xf4vW*:\xd9\x83e\xc7r\xbdh\x15\xb2.\x02\xae\x10\xa5\x81\xbe\xdd\xa1\x0c,`\xe5oV\x8aC\xe6\xb2\x0b\x1fU\xb2,\xec\x94%F@\xa3g\xee\xbb\x93\xd3j\xe8\x10\xa7\xfda\xa9L\x9d\x8c\x85\xb7\xebW\xdd7\x93\xd8[\xb3x\x1a\x14\'\xecs\xf5\xe6\xaa\xfb\xe6+\xefq^\x11D\xf3\x12\x9b\x8bI}\x15\xa4m\xc1\n\xd26\xc3\xfbM\xd3\xdd\xb1f<\xa8\xdaPP\xf0f\x86\x8a\x1e\xfaML\xef\xe6\xbdu\xcb\x1aJG\x8e\x80R?"\x8fOm\xf7\xc5o\xb0[dq\x19#\xa3\xa4\xd6`\xc5\xa2\x1a\x84V\xac\xd0\xc3\xe5^\x1eeo\xb1\xc5\xcdE\xaf\x8e\xef\x83\xacf\r\xacZ\xdd#\x97,\x1b@\xd5x\x0c2\x939\xf2\x0c6\xad\xe8\xd2b/\xc0a\xad\xd9l!\xb2\x14\x16\xe3o@>\x89A\x11$m\xd5\xc0"4\x10)\xfb.\xda\x12\x99\x8b\xc9X\xe4\x04\xab>\xf1\xc3\xef:H\xbb\x96\x1c\xda\xb0N\xca\xd8\xee\x90\xe8"I\x95!\xd3\xce\x98\xd8<\xa7\x16\xe8soX\xa2{-\'rZ\x01\xd1x\xca\xa2fKj>\xf3\xaa\xc8\xb8v\x07k\xc8\xb8\xe3L\xabz\xe4\x9dso=\x10\xb6\xca\xbd9\xb8\xa1)F\x80\x18M\x88\x17w\x1bJ\xd8\x1eoQB~\x82\xfb\xc1\xd9@\xa3\x19j|A.\x88\xaf\x05\xd7\xcf&>.\x897\x94\xde\xdaf\x142e.\x92\xd46\x97\xfa\xcc\x00\xb5\xfb?\xdf\xdf\xf5\xa5\xee\x81\xc0\xfe\xc9\xd8\xa9\x98\xeb\xc4\x89\xf0\x9f~\xe2\x8ec\xa4\xe7\xa6)-\x8d\xd3g\x04\x91k\xbd\xc9A\x1cX4\xe3\xc0B\xcf:\xeas\xcc\x02\xae\x94o<\x00o5\x87\xcf\xa3Q\xfd\xbe6\xff\x8d\x19\xef6\xff\xc3c\xb1\xd0\xbaI\x13\xa3\xdcZ\x9f%17Q\x00qQ\x16D7\xdb\xcbM.\x88@\xd4\xbd\xd3\xd5\xebK\xcc\xcbRb\x8b\xa2\x81\xc6\xf8be\xedj\xf5\xa73\x95\x8dY\xdd\xe9\xc0\xe6*\x1a\xe4\xe3w:Bj\xea\xc0[\xcfPM\xa3\x8fn\x90\x05\xd57\x12\x9b\xe4\x8c\x84\x12\x91Q[_s\x1f\x7fm\x8c\xca[\xc7\x0c\xc5a\x91`\'\xb8\xa8\xb6\x95\xac \xe6;\x10>D}\xa9C]A\xa6\xcc\r\n\xe7\x91\x061\xc8\xeep1\xb8\x0cS4\x8c\x12\xc6\x87;\xb7R\xfd\xf6\x99;9\x19\xa8\xa6\x08\xe7\xf7\x16e\xc1\x8a\xbb\xe5@\x01Q\xea\xab|j\n\x8aI~g\x8c\xe9\x1c\xcc\xb0)\x96\x05\xdaF\x96\xa5,\x80\xe2\xdd\xe0\x8eP.r+\x9em*Lk\xe6[\xff*\x0e\x80\xbf+\x0cu\x9e\xfe\x94O{(eg\xc5\x07;\x0f$R\x08\x14\xbe\xde\x953\xd5\x95Y\xa6/o\xeaS\x10p\xa2\x14\x87:)KQo%r*\x0fc\x8da\x8c\x19\xff\x1ez\x97_\xda\x84\xf3\xd1\x10\x18\xaeT+H4\x94\xca\x8b\xf2\xac\x02\x8e\xd7\x0c\x8e+\xf1\x88G_"1\x92\xd8\x98s\xd38\x08\x8a\xe8\xb6\x99R\xa8\xbb\xf9\t\xd7/o\xc9B\xfaP(N\x95)t\x00\x0b\x99\xb9\x91(5\x17\x92u\xed\xa0Q\x17\r\xcf@GD\xbaM\xc4\xa7\xbc\xc7\xf8\xeaV[\x93\x83\xe4@\xef\xe1\x86ZI\xfd\xc1\x0c\xda\xe6\x0b\x8f}\xc3eD=jpJ\x03\xd5\x9eg2@\xe5*?\xc6h\xf7\x98\x13GPz\xf8\xe6$\xb8\xf0\x02s\xac\xa3}<\xaa\xe2\xdf\x93/\xdd\xd3\x8b\x81AW\x1c\r\xf6f\xbdjE\xf2[\'\xbc%Y%\xe8\x95\xe4\x15\xfa2"!)\x8c1\x94\xe2\xda\x14-\x12\xe1H\x8c1\x19\xdd\xf2\xc2\xe7^\xf4\xdc\xff\xa2\xb8\x99JE\xb5\xa0+\xffT\x84\x84\x14,\x01\xe0\xda\xe9-\x8a\xabc:\x01!\r\xfb~\x8eBVU)\xe9\xe4y\xff\x9b\xc6\x83K\xb2\xfdAU$\xfb~\x8b\xf5\xc8-\xb6;\xe8x\x98$\xfe\xd9\xfd4d\xffir\x04<\xc6\xdcIG\x86RT\xfcN\xec\xab08\xcf\x8dc\xa9\xb3\x9a\xa8\x02\x0eo\xed%\x96DV6\xcfV\xd7\xe6u\xcb\xcb\x0b\xd3\xc9\x10\xad\xf2eY\x01\xcaY\x7f*\x1d\x1d(\xcf\x07\xa1\x9f\x13\xad\xb2j\xfc\x95\x0c\xf0\x14U)4=\xac\xdd\x0b\xa1\xd0"\xf4\xa6\x0c\x99oA\x8e\xc0! 4Y\xc3\xbd0\xe9\xce?`w\x9d\xa7\xeaz2\x9b\x1c\x86\xb0\xf4\xcb\x90\x99)\x13\xdc\xf8J\xcfe\x8f\xd3uo\x05A7\x9fB\xb4\xa5\xe1|\xbb\xc5\n_x\xe6X8#7\x9b\xb7"\xc9A\x08\xf6sT\x90\x81\'$]]t\'\xa9\x16\x81\x0cw\xec\xc8\xe7\xca\xef\xd7?\xd4o-\x07K\xbdiSZE\x1d\x80\xd6\xb56\x13\x82]t\x96\xa8\xa2\x10\xbaHS\x81\xba\xb45\x05\x1f\xa4\xc2\xa7v\x96\x04A\xc9\xca\x8cp\xe0\x14\xf7\x80z\xc6\xbd2\xc4\xb0\xe0\xa9\x08\xa0"\xb8\xcf\xe7a#\xf4"|\x9d\x9e\xd0\xb0\xa70\xaa\xe8-\x9f|\x89\x93\xbchz\x92\xbe\xc80\xf8dh\xb5T\xe4E\x1d\xef8\x11S\x94\x87\xec&\n\x1aU`\xd2\x08\x85F\xde%\x9f1\xa6\x1b7Y\xd4\xa8L\xa7\xaa\xff\x99\xe94$\x08!H\x9f\x8a\xf5\x94\x0f\xe0\xc8p\x84r\x00\xbc\x1d\x95/g\x8b\x9fI\x1a\x8b5\xc7\xbd\xc6h\xc3?n\xf8\xbdYM\x18=\xb6#\xad\x1di\xfcH \xd0\xb7\xcb\x9a\xdc\xb2E\xc5\xb9\x8a\x13\xb5\xda\xcbf\x03\xc8\x9d\x9c\xa8/?\x11\xc3y\'\xefneJ\x887\xe8\xf6\x0f3\x99\x16\xee;\xf1.\xb3\xa77\x97\xcc\x94S\x96\x12\xc1m\x9e\xa5F-\x06#\xd8\xe9\xde.\x8b\xa4\x8dA;\x05r\xcfo%<\x9eb\x93\rK\xb3\xd8d\x9b\x9e\xaa=}]{\xfe\xa4q\xe5I\xfd\xbb9\xb1\x86\xc9\xe5\xc6W\xcb\xb5g\xb3\xb8*5,\xcfg"H4\xb1a\x14\x8b\x97s\x00H\xf47Q\xfc\xc4\x8b\xa5\xc5\x80\x9dR.\x08\xeb\x08\xf7\x8d>\xb33\xaa\x19O\xba\x89\x14a\xb0\xc1\xcb\x0e\xac~\xbf\x8d\xd3\x87\xbc0_\xd1\xca\xe6\x87p\xd6\x07\xd6\xfe\xe2>QB\xc8\xe49\rjkq~\xb7\xa3\xdc\x8d\xea\xb9M,\xb8!\xd1R\xbf\xfd\x00\x03\x80\xe7\xb4\xb1(?\xben,\xde\xc1\xac\xed\xa5W\xb8\x8dr\x9evp\xf1\xe5\xdc\xc9\x1c\x86\xabx\xed~a\x19\x03\x8d(\x91\xe3\x87\x1f\xcb\x98\x87\x8d\xd4\x08\x80j\'\xa5k\xc0\x16\x9d\x93\xc9\x83\xc0\x0c\xce@\xa4\x1e\x99\x80Y+\x1ac.\xbf\xd0\x87\x1c\xea\x96\x91;s\xb9:9\xc3\xdb(+\xf6\x91Jy\x94\x80\xfb\xe4rca\xc6\xcb:\xcd}S\xfdq#$\x15\xd6L\x9bd\xa2\xb4\xb6\xd8\x89\x00\x83\x06\xa3\xc4C\xcbd\x1an \xfb\xd0\\\xf3\xef\xd5Y\xeaMx\x84\x1co\xb2\x85@\xbec\x95|J\xa4v\xf9nc\xe2"O\xe6\xc1Z\xca\x8e\xb7\x1b\xd7\x7fk\xf8J\xeeh\xb5r\xa8\xee\x06\xfc\x805#K\xda\xadC\xd9\xf2\xad\xed`\x8c\xb5,\xb4\x1c\x8bp\x15\x03Y\xb9\x18\x7f\x8e~\x1a\x1b\xd3\xf5\x92>\xc9\xd4\x8f ]\xa67\xbf+\xa4\xb0\xa4\xf9];[\xbdK\xceB}\xef7\xb6.0\xd9\xde\xdb\x7f\xc1{C\x92\xf3\xe1\xef\xf2\x8b\x06\xd3\xceps&\xeb\xe7\x80\x9a"\xc1\x0ff<\xfc\xb6\xe8\r\xb3Q\xac\x14\xd43H\x1eggA<\xe8\xde\xb7\x88\xbfi\xd9\xce\xca\xea=l\x98\x01q\x12*\xf15\x1bX,\x13\xd4\xf1\xaf\x9a\xd4qt\xb4\x99\xd7\xc6b\x86\xde:!\x9d\x1de\x17\x8b\xe7\xc5\xc6~%\xc7h\xda\xb9R\xc8\x08{\xe4ID\x05:02\x00\x83Q\x8c\x9c\x83\x92Ev\xa4Q\xd8\x91+\x92\xeb\xe3\xdf\xe1\x84G\x99\xdbes\x84\xc7\xd3\x88\xe1\xf7\xd3S<\xc6\xa715\x17\x1cc3\x8a\x9e\x8cw\x1a\xc1\xbf\x1b\xa3\xd4\xec5\xf6\x1f\xdf\x8dr\x93\x17]\xf3\x0f\x82\xc2+4+`\x8cG\xcc\xe1\xe7\xa69-\x1f\x8a9-\xf0%9&<\x1c\xdb\x18\xd5\x87eD\xb0\x03\xb4\xec\x0c\xca\x9c$Un\x8096\x90\xf3F\xe6\xfc\xc9\xea\xef\xe3\xda\xeed\xc6\xa6\x10\x01\x8e\xa5\xb1K#\x1c\x99\x06*\xf1\x8d\x8c\xc6idw\xbc\xd1\x81\x94NU\xf5L\xda\xc3\xdcG\xb7\xab\xd3/j\x0fV\xea`MM\xdf\xad\xdd^Wagz\rW\xc1\x04\xd3\xc1\x0c\xe2\xd1"\x06\xc6\xbdg\x89\x98\xab\x9dA\xa8ds9\xce\x17F\xe9\xa7\xa8Z\x0c>\x84\x8d\x1b\x9f\x817\xd6\xdfl\xd4_\xdd\xd1!\x9a[\xc1\x00\xf9\xf8\x8f\xbe\xe9\xd0Te*FL\xd1\x10&\t\xe6\xb0\x13\x9ckH\xa9BA0\x80\xb9\xb2Ec\x9e\xf2\x00\x1e\x8eg\x14v\xae`807\x06\x8a\x00\x8d\x01\xdb\xe9p1)\xd2\x135\xc1\x8a\x930(Z\xc7F\xbb\xca\xaeU\xd7\xe7\xaa\x17\xcf\xb9\xd3\xd7\xaaK7\x1a\xe3?\xbbk\xf7*k3\xa2E\x16\xa1(\x15G\x82\x1d\xed2\x85\xc9i\x84T>YFoC\xf3\xe6\x15(<U\x16\x8cb\xcb\xcc2\x9bs;\xbd\xe7\xaf?\xfa\xbe\xba\xf0JC\x99\x0eBe\xf5g@P\x90\x9f\tK\xeal\x95x\x92r\xf3\t\xcf\xa5\x01\x0e\x18vv\x7f\xf8ag\xd7\'\xf1\x0e\xf8\x7f\xe7\xee\xd1\xc3\x87\x8e\x00\xf5\x0e;\xff\x83\t\xa17\xd5=\xaaHbLK\xfe}\x14\xb0t>\xc7\xbe{j@\x00\xa9[\xa6Q\x7ff\xca\xc1\x11Ub\xb6%q6F]3\x16\x1f\xd9P\xb4\xb8\xc1\x95\x14\xe7@9\xe3\xf5\xa9s;>\xbaT@\xc9\x0f\xcf\xb9\x9b\x93\xc0\xe1\x8d\x85o\xd10\xc0\xc4\xd8\r\xb6\xef\x84\x0e\x94\xe3;y\xd2imf\x1cl\x89fD\xc4\xa9\x87\x95`\x8fv\x01\xec\xd6)jh\x8b\x1e\xea\xeb)\x16\xf3\xc56\rM\n\xc3=t\x0ff\xdf\x9b\xb7\x15\xdc\xf0\xb2>B\xcc\x1b\x14E&\x90\xdaV\xdb\xd1\xfb\xf4\x85EK#\xa3\xe0\xbf\xb8,\xdaD&4\x1e\xf5\xc93\x01\x14\x0f\x88\x1c.\x8d#\xd3\x97\xe5\x03\xe8\x0b\xb3xRPP\xb9\x80\x11\x05pe\xf1\xc5^>\x84\xc7\x0e\xe3i\x13T\x08\x941\x0b`\xb1\xb4#F\xb0E\xad\x901\x05\xeeAOP\xf75\x12\x15\x87-pB\xd8\xa5W(N\x16\x1eUV\xc7\x19`w\xfa;\x96\xb4\xb5\x17\xdf\x82\t\xc3\xfb\x13J~\xcc\xa4\xfe(\x02\xbc\xceFl\xe1\xdc?\x12\xa28~.\x89qw\xaa\xb4\xc8dp\x8c\x19NT\x95\x99\x98\xfb\xf7\xab\xdf_\x04\xe9Vy=\xdeX[\x17G\x9b\x80\xbaY\xfa\n\xdb\xd6\xe76yRgmq\xd3\x9d>+\xa6@\xfa\xda\xf7I\xdcG\x18\xe3|\xafd\x9bY\x16<\x98\x9d|<\xc5Kj\\\xb9X}\xf4\x1c\x9d\x0c\xff\xda\xc2\xe4u\x9f%\xc6:*A\'X]6\xc9q\xe1\x9dr\xda\xe4Tlx\x19S-\x08\x9d\xca\xda\x1d.\xae\xf3&2\xfb&_\x03\x1d\x15\xc0\x1f\xb5\xf4\xf4|!\xefh\xf9y\xd1\x0f\xdcFg0u\x8fz\xbd\xcc\xc2\xa5\x8e(\x16\x86k\x1e;{\xdc\xdc\xd9\xd5\xa1&\xe8\xd1\xb0@;\xc7\xd6\xad\\\x96:\xac\x80d\x08\x0f\xba\x05\x99\x9a\x01&\xc7\x0f\x8a\xdc\xfb\xcd",N\x9b\x9a\'\x9al\x88\xb21\xf3;9\xed^\xbd\xee\xde|"\x9b\xc1h|\xed\xfd;\xee\xe3\x8b \xfb1\xa1F\xe4U\xbd}M\x0b7\xa2="\x91\x10\xcf\x9f\x0c2c\x06\x03\xce&\x8e\xe1\xa6\x1eiO@\n\x90\x8b\xa6\xed\xc8\x99\x7fT\xe0Sl\xc71\xf3\x86\x85\x8c/\xa8\x9e`b\x81"D\xffwK\x8d\x85slVys#\xd4\xb3C\xfc\x8fa\x87\x8d\xb6\x12\x91|K\xff\x01P&z\xef\x85F\xf7\xc4\xb1\xd6";5S\xbd\xb4\xe9\x93\xf5[\x85\x8f@\xbc%\x11\x0b\xd1\x80PV\x15M^\xf3\xe6\x10\x87\x1cQ\x02\xa7P\x1cdd#\x05\x950y\x14(S\xd3l^\xf2.\n\x81\xfbtJ\x9c\x82\xe0\xcb;\x00\x92-\xe3\x08\x8eB\xcfZ$R\xa3j\xcbp\xb8\xfe\xf1\xdd\x1futt\x9cPT\x96\xb5h0\x84v\xb0Q\xef>\xd4\x16M\xaa\xd9\xc0\x11\x90\xac\xcay,\x03\xf5(\xf2to\x06\x9a\x19\t~\xef\x95\xc1<\x9e\x12\x01D\xb6\x83\x88\x1e=\xe2z\xf9RK>\xf1\xdb\xd9\xe0Q\x10sY\x8b<\x06\t\xf9FT\xfa\xf4a\xe2\x00\xd9F\x12\x94\x08!\x91\xc5\x912\xfe\x92\xef\x8f\xef\x10v)\x9d\xdf\xe3\xbe\xfc\x99\xabw\x02\x90\xb0\xd5*a\xa0t\x04\xbc\x1fq\x17\xcf\xe6\x81\x86\xf29;)\xebj\xe44?\x03\xa98\xa3\x8d\xe2\xd4\xb9\x8e\xbbo\xe1\xce\x0c\x0bTR\x06\x0e9\xaa\xc0\x90\\WG\xf2\xdb\x9b\xc4\xa9X\r\xf5\xeaW\x93\x82\xca\x97n`\x86^\x9b\x7f\xccZ@\xa1\x0b\xdf\xd5M\x07\xb2EK\xf9\x92\x99\xe9\x16s\xb3R\x96\x93\xec\x8eT/~\xd3\xb8r\xfe\xed\xfa\x9cW\xe5r}&\x82\x1d\x88v\xa9;"J3\x80\x05\x86,]\x8a*:5\x8d4U\xc6\xc3FS\xc7l\xd1J\xe6\x8b|>\x9d69[\xb3"\xc8\x82 \x87\x80i\xa1\xba4^{.\xbc\x81\xda\xb3\xf1\xc6\x853\xe8\x01\x81a\xb9\xb0.\xca\xa64[A\x89\x18\x9e\x10\xe9\x99\x93<\xd2\xbbh\x15\x10\xf1)4\x18\xf3\\\x13"\xe1Rc\xf9=\x0e-Z\x9c\x8dI\x93\x8c\x16\xa3#1\x86\xe5\xdd\xc2:\xa7q\xf3u\xed\x15\xd6\x84\x88\xd1+\x13\x17q\x17n\xdc\xad\xff\xfd\xb6\x0e\xb7\x04\x8e\xf2\xde\xe2\xbd\tZ.\x80\n\xc6\x1aC\xba\x0f\xe3\x95 N\x8a\xe5\xd2 \xa2\x86\x02\xef\x0e\xe9Q\xce2R\x19\xb6d%Y\xc5\xc4*WP\x0f\x17{\xcb\x1erC\x1c[4\xfd\x98)A\x9c\x07G*\x11-\x81\x9f\xbf\xc11\x05L\x18\x8b\xa7+\xafg\xbd\xa6}ey\x11\x9d\x0c\x82\x98\xe3<\x10\x0e;\x8b\x04\x8a\xa00h\x86\x93\xc4"L\'\x86\x9ct\x88\xa3\xf6"\x82\\\xb9\x86\x84\xea-"z7\x1e\x8fO\x8b0\x8dVV\x1f\xe2\xefh,YD\xa4\xb1~\xfc\x01^BWq\xa4X\x84sZ\xf2\xaa\xefA\xbb\xe8W\xf3\xd3\xaad\x85~\xf5\x11]=\xf33\x86\x9a\xf9\xaa\x98\xa3\xf5\'\xd3.\x89\xd3\x1e\xbcX\x9dH\xc2\xb5\xa6\\\x12[,\xb0\xaa\x17&jk\x17\xd8\xa6\xf5\x0c{P>\xa7\x9f\xea\xc7*1\x15\x8b\xa91>jhv1\xe8\x9c:\xd8Kd\x0en\xceF\xf0\x04\xb7\xb4q\xd57\x0fLb\x85\xd7F\xb5\\\xac\xf7E\x80\x91\x80\xaa/^\xae\xcd\x9f\xad]}\x06[\x8f\xe88\x7f\x03\xe7\xa0\x83\xd5u{M\x1c\x8a0\xbd\xd0\x18\xdf\x14R\xcbg\xe6\rb%6\xb2s\x0b\xa5\xf7\x05\xa8<\xf4\x00H:\r\x9b\x99L;h+,\xae\xb2\xb3\x9c\xe0\xc4Xw>m`]h\x96\xad\x0b.\x97\xf4HT\xa5&\xf8T4"M\xdd<\x13^\xe3\xe3)\xfe\x08\xe0\xd6.\xbf\x10\x07>\x04\xad\xc2 x\xc2\x18\x0e\x91\xbf\x98)D\xd1\x1c\xbe\xac?\xd0LoTN\xed\xb2x\x02\xe4\xbf\xa3j\xc6D\xc0\xfe\x8f\xe6\x00Jm\xc7r\x1c\xbf\x8e\'/\xa7q\xf9%\x83\xcc\xaa\xc1Sa\xe4\xfd\x08\x8a8s\x0bnf\xaa\xd9"\x83$\xd7\xf1{\xe3\x93\xae\x8e\x8ew$\x89d\\\xde\xaf\x80\xb9=\xb5DG\xf1$q\x04\xf4\x88?3\x14P\xb8\x06N+\xdb\xbc\x00\xd4#\xe4\x97\x9a\x15\xbc\xd5\xab\xd9T8\xca$\xc8\xa6B:\xc2\xa2\x8f\t\x0c\x8cq\xa5x\xbbG\xd5\x9f\x98L\xc5z\x08z\x95!L_\xc1o\x1fj\xb7EZ\x16K\'-\nOkR[\x88F\x9e<\x03l\xda2\xc9\xae\x97{\x8b\xba\x05\xce\xaaP\xdd\x02\xc7\xbd\xbc\xe7b\xda\x97\xab\xc74\'\xcf\x0b\x16\x84A\'!\xd1=\xa6\xe6\x15\x00\xa3h\x9a]\xb0^\xa81\x89\xb9\x06\xcct\x19\x83\xc0u\xa5\xc1\x91`\xf6RC\x828\xb6\x81LL\x1e-\xce\xc0o\r\xf0V&o\xd8\xb6\x93\xa0\x12\x82\xc7\xb0\xa4k-\xa2\xe4\xb0\x89\x85B\xf0\x044\xfd\xd7\x1e\xb4\x0c\x1bS\n\xfa\x80S\xe7\xf8\x18\x89\xe6\xe1>aPx\x02t\xab\xbb\xfc\xe4\x12\x88\xd1\xe4\xa9:\x1b\xeb\xd0\xb1D\xd8n\x86\x95\x84\xea\xc5\xbb\xd5K/XG\tCkzAW\xa6\\\xa1\x1d\x04\xb7\x150\x1eu\x0cc\xa499\x14\x8d\xf9W!\xe2\x19!Q\x0c\xc9h\x12\xe3\xdb\\\xfe\x1e\xb0im\xe4\x10\x94\xc4\xed\x18\x8e%G\x0f\xd5:o:M\x02\x92\xe7FjG\x984a\x82\x8f\x17b{p\xe9\x06\x88:6"\x98\xba\xdc\'?1\xf9U\xbf\x9a\xac>\x9d`9\xce\x19tv\x1a[ah[\x8bP\x81\x1ai\'\xa2\x84V:Z\x9e\xbeB\xb5\xfb>\x95\xe8\xcb\x86k\xf4\x86\xc2\xee\xd5:\xca;>\x11\x90\xc5\xb3\xd2\xd0\xdb\x01\xa9`:N\xe8\xd6\xbd_(j\x1b\xd4\x8b\xf2\x1b#\x00I\x0b,C\xfd,)&cl7\xc8Y\x18j\x01{\x1aL\xb5\x94\xed\x08\xe9\xa1\xf7\xe6(m%\xe2TW\x1a\x13\xdf2\xbf!:\xae^o\x8c\x8f\xa3\xadH(\xa8\xaf\xdf\x07\xc9\xac\x07p\xc3y\xf2\xbd\xb6P\xf7\x98\xd1\xc3\xa7h\xd6\x16\xae\xb3\x02\x98\x1cx\xa2\xa1i\x7f\xf0\xbb\xd9\x97\xde\xea\xfdA\xdf\xfa]\xf7\xef\xb3\x19\xc4t\xb9H\x05\x7f\x1c\x07U\xa9@\x1d\xcb\xaa\xdb2\xf8\x0c\x0e\xc2\xa9\x03r\x94\xe7\xfa\xbe\x90o\xa9L\xa8\x04\xe6\x7f\xf6\x1d:h\xa0\x7f.\x93\x1b\xe2HZO\xe6ao\xc3\x7f\xe73\xa5\xada\xd9\xce\x88\xc1\xdb~\xb0w\x8b\xcdj\x85\x9e\xc8\xd6\xbe\xe0p\xac\xe9A\xe8\xb9\x97\x0b\xab\x88o\xce\xd7\xefm\x88z\xf1\xd3k\x95\xb5\x0bpCec\x96o\xd8^\x8d\x8c\x10\x88m\x181\x8f\xbdC\xf3\x1c6\x8b\x18!R\xca\xdd#\x05\xe2\x84@\xf1\x0b\xcd/\x0c\xb9\xb7Y\x1b\x86\xbd\x8by\xab\x9b"qq\x94\x9a\x8e\x045\xb65\x90\xaa\xca\x95\x9fP0G(\x05\xd6\xd4x\xa2\xca\'NZ#Np\x1a\x9e\'\xaf\x18Z\x89r\x9a\xce\xcbv%5\x9ap\x1d\x05g\xdc\xb9\xdd\xa4\xb9\x813\x0cHjt\xe9\x16h\xa3\xc2\x0f\xbc\x12\x89\xa1N\x1c\x1d{\xd7\x02\xa9jT\x11\x968\x1a+O\\"\x18\x1a\xab-Z\x18\x00\x9c\xfdn\\\xbb\xae\xe7\xf6\x14gx9\x1avM_\xbe\xac\xac\x9d~\xd7j\x80\xa2\x10|^\x88\x17\xea\xa1"+\xf5q\x9b\xf2\x96\xb3b`\x8f\xe1\xfcF\x8c\xe6\xe4\xd3\xfe\xb80\x85\xae\xf04\xf6\xa6r+\xda4\xca\x87\x89\xd1\xa5\xb2\xe0\x08\xc5\x96\xfa}\xf3\xc8\xa9\xad\xc0Aw\xc0\xdb$J\xd1\xd2\xb2"\n\x06.\x7f\xf0\xd6OeFa\xda\xa5\xd5+\xa8\xd4L\x1eK"\x02Oh\xa2\x14e\xa13\xd5\xd9#\x10\x1c.\xf6B\xfc\xf1\xd0g\xe9\x87\x94\x88\x9a9\x19u\xc0cd\x16\xafV\x7f\x9c\x119\x9e0\x8fn;\xe0b\xf0\n\x13A\t\x0cV\x15m\xe2\xa4(G*\x08\tX\x92\x86\x189\xce\xd88\x81\xe3\x81\xd4n\xe0\xd0\x84\xa2E\xe7\xf6t\xf3q\xab\xdb\xa3\x0c\x1c\xbe\x9dO\x8b\x03\xb0(\xf2"S4J!\xcb\x16,\x19L\x80k\xd90\x93R\xf4\rq\xb4Z\x05a\xa8\x89\x8c:p(C\xa8\xbb\x1c\xff\x10\xb9P\xc5\x93d\xd7\x08\x9fQ@\x7f\'`\xa3K\xefO,{\xb1\\\xc0NJ\x07\x86\x0e\xb2g\x84\x88\x91]\xdc\xfd\xcc\xdd)\xe4\x97\xb2\x0c\x0c\'\x17\xf7\xf4S\xac\x00\xd3<\xab\xda\xd5g\xd8B\xbf\xb4\xccR\x0fu\'E\xb1\xbdC\xde\xdf\x8bV\xa4\x9c\xea&&y\xbf\xa5\xf6\x9c\x12\xf3\xf1\xc4Zyc\x91\x05\xda\xa9\xf8O=\x1c\xc5;\xa6\xe4\xb8\xa2\xd7\xe0F\xcb\xf0\x05s\xe1\x97\xc8[\xf1(\x13\xfe\x11hZwrU\xcc\x0e\x9f\x17\xa5\x89\xbfl\xd1-\xc5F`\xeb[\xec\xfcV\x8fV:\x8e\x1cWr\xb6\xa5l\xc3\x8e:9j\xd8?*\xb4i\xd3\xe7\xee\xb3z\xab>\xbc\xa3\x89\xc6+g\x987d\xa9m+\xd2\xdf\x0e\x8c\x81}\xf7\x84\xa3\xa3wc\xd1q\x03\xaa\xd8q\xbb\xcf\xef\x1d\xc8!\xcd{\xfed\xd6.\x95\xb8xN:X\xa0\x01Sx\xe0\x81V\xe5\x1d\x8a\x08\xa5\r\xb5\x85\xeb\xda\x10\x07\xcaOz\'D\xfeb\x91\t\x14\x81RQE\xe2<]\xd1f4_$\xf5\x19c\x92\xc1(\xed?@&ZrE\xc4\xfbE\xa7\x0fe\x1d\x02\xc7<nE1\xd8F\xe3\x8b\xb5\xde\xc4\xf2W<n\x98\x94\t\xe6yW7\xf4\x84\xc7/\xa5\x1cQ\xb6\x9d5O\xd9\xd92&\x14\xfc\xf8\xa1\xafQ\x89\xf8/\xd3\xd8\xfc_\x86\'\xd5> \xcf\xa5&9\x8au\xa6"\xf3\t\xbe\xbb\x9d\x1e\x91\xa9*_\x9e?\x14W\x93\xd3\x9c$\x11\x01\x0b2\x0f\xb1\xa6\x88\x06\x13\xf3\x99\xd4\x18\x83;;\xcb9\xa1\xda\xe5\x07\x8dk\xb7\x7f)\xbaBk\xde\xa3v\xae\x84i\xf6\xbc)\x0fU\xd0m~\xc6l\xd8]\x12\xe9\xbf7:\xb6 \xbaw\x81Duyy\xaay\xd5\xd42\x88\'A\x88Lz\x088\x8euM\xa7\x11\xd5\xca\xb3S\x07\xe3m\xf5t\xd1;\xe8\xd7\xe4<8\x93\x89\x0f\xf1K\xdd\xd4B\xa5\xb39\x04\x04\xba5]nged\xfd\xf0\x12\xba\xe9\xe0<\x81\xb4\xd8?\x86\xb0c\x05\xd5;\x8f\xb9Ay\x0e\xb8\xcaD\x12Q\xb6\x199k\x88\x82-\x18g\x11\xf5\xb2"\xdeom\x8d\xb2\xca\xe6\xb5\xea\xb7WD^R\xcbHz\xa2nUd\xe89\xa9Z\xbd\xb8\\Y\x7f\xfcOAVN\x9c\x8b\xd9\x01\\m\xe7\x90\xeaD\x1bF\x9b\x86\xcc\xd8?\x88\xbf^\xaa\xe8\x13}\xe8\x1c_3\x93\xc92W\xd1m\x8d\x1a\x1e\xd0U]^D\t\x0f|\xca\xd1F\xb2\x9a\xff)\xeb\x0f\x9a\xca\xbe\x9c[+[8lF\xba\x85\xa6\x8f\x83\xad\xf9\xe0\xe4y)\xb2\x01\xb3\x98\xca\xa0"\x04g)\xbc\xa0P\x9b\xb1\x02>\xc0\x83\x05=j\xcd\xc7\xf1R\xf0\x10\xdd\xbf\xaf6\xf4\xe20%\xf1[\xc5\xe8Z$O\xda\x8d\x90\x84\xeb\xef\xbb\x8d]\xbb\xdf\xe5\xddR\xbe\x8f\x0f*w0\x1d\x9c\xd4\xd7\xfa\xd7\xb2U\x1c\xf1\xca\x11SeL\xc9\xf1\xf4ZQ\xd1\xe3{\x1aSxe\xf5<x\xba\x9c\xefPi.\x915\xa6$2O\xc4\xe6\xbaEl3{\xf2N\x9774\x9b,\xd6\xfe\x8e\xae"t\xf5\xc4\xaa2\xf9\x01G\r\xeb\x17\xe7F\xa7p\xcc\x18\x8d\x827\xb2V\x16\xd7\x8cc0\x9c\xa6\x90\xb0\x90\x80T\xe1\xe4^\xbd\x89\'\x0f\x9c}\xd1xp\xc9\xfd\xe1[\xf7\xfcUu"\xb4\xac\xe3\x1d\xaf\xbf\xf9\x1a\xed\x1aJA\x1a\x9f\xe7\x07\xb6\xe1\xd4s\xbd\x10\x9d7\xbdM\xef]\x15 \x88\x05r\x99\x10\xaf\x13sb&\x05\x19)\xae\x97\xb2M\xb0\xe2@r%C\xac\x0eU\x84\x00P\xf3\x12\x99\xf0T\x9e\xac\xb1x\x07\x96\xb2]\xb7,\xa4\xa8&\xe2\xaf\x11\x93\xf8\xba\xf2\xe7\\\xc4\xf8\xc0\xf8\xfc\xd0\x818\xae<\x81!\xbd(\xe5{\x9d\xee\x08x\x1f8\xf88\x12;\xde\xbe\xeb\xa3\x8e\x8e\xdd\'\xb6\x17\x8a\xb2\x1d\xef\xa0\x89\xb0\xe4vX\xd8\xc9wr\xb9,v\tOgo\xb9\x89\\\x91\x11>\x8d&,\xe5\xe7\x95\xb2}\xac\xa6\xd2\xd0\xaf\xd4=\xb1m4\xcd\xc92\x8f<Vt"qS\xb6\xc0\xf4\xb2\xfba\x0b\x17\x02\xe6\x12Uop\xf9\t\xad\x116\x9c\x8b\xee9,\xbe\r\xba\xc5Uo\x93`\xe5AI\\\xef\xc1\xe7e$e\x80)\x9f\xe3T\xbfW\xb4\x971\xb9bIUa7\xd9%K\xcb\xdc\x97\xc3\xa9\x7fLl\xf2\xf8?\xe5O\xca\xc3\xcfE\x0e+\xb4\xd3\xaf\x15\xb4\x02\x9an\xe3x\x96\x0bR\xd0\xc3\xc7\xd5\xb2\t\x8c>e\xd9\xe7^\xc9\x8a\x15\xb8v\xfc\x04M{\xc8\x1e\xef8\xa1\x07\xb0\xf2\xb9\x04=5rb\xbbi\x04\xf2\xc09\xdd\r\xe8r\xe4,xQ\x9dL\xb1\x18\x1c)jk\xf3{ZXo7\xe6\xf1\x98cnf\x92%\xbbl\xc9Q\xd5\xd9\x85\xea\xe5ga\xa5r\xdb\t\xc2\x10\xa2\xde7\x08W.\xa6MA\xb0\xf4\x00\xcf&\xe5\xb1\x1f\xb0D1\xc7g\xab\xc8\x9b\x1a\xe0\x81\x15\xfa\xb4,\xcf\xabl\xb9\xdb[\xc1\x15"\xbd\xd0\x95\xe45\x1eo\xef<q\xbc\xf3D\xcc\xab\x0b\xdcV>O\xd9\x13,\xa1\x95db\x12\x92\xf4\xce\x9d;\x1cv\xf4\x9cG)\xbd\x88\xf0Z\xa4\xf9*\xebk(X\x17\x97\x19\x1b\\\xd3\xe4\x05\x97\xcf\x9fG\x81\xfb\xf06\xe6r\x19EW\xce\x08\xfe\xbf\xfc\x02k^E\xe1\xcc\xf66\\\x1e0\xb7\x05#\xf0\x95\x04\xe0-\x123\xfe\x1b\x92?\x11\xbd>\x84@\xff\x9f\xd7\xaa\xce\xc2W\x92\xe7\x95Q\xf5\x92_\xf1#\x7fE/\x19k\x06j\xbb;!\xd0\xca\xe1\x1c\xce\xa4\xcawKD\'y\x8e\x81\xb3\x85\xd8\xe1\x9c\x91jf$\xc1)$\x0fy\xe7U\x10D\xcf\xe7\xab\xd7n4W\xbfl\x05]?(\xc1\x93\xe1&\xa9pW\xe8\xe0\xf3\x12it\x15\x8e\xc1\x96E*\xa3\x94\xc6 \x15(q\x8a\x1bm\x01\xf48)2\xa3%\xbd@n\xce\xd3\xd4\x9b\xdb\xd7Te\xbc\xf0\x8e\xfd\xe5I\x82\x82\xa8\xd6\xaa\xfa\xec\xbb\xda\xb9\xe9V\xd6\xa82\xbd\xb1\xc8;m\x9f\xc2\xddG\xed\'\xa1\xfa\xd0\xf8\xb8cw\xbc3=f\xe0q0\xf3\xdfo\xd3\x16?\xc2\xf7\x88\x83>\xc4 \x01\x1e\xc9\xd1N\xf6\x81/\x95\xe7N\xbf\x845\xb8\x17o\xa8U\xb1\xa7\x85\x17o\x83\xd8\xfb\xb115\x87\x93"&V@\x1bl\xb9\x12\x01[\xb4\x05X\\kY\xc2\xa3\xb5\x94\xfe\x15\xa6\xa5\xd6\xfc\xe4\x95\xb5\xd8\xbe\xbe\x06\x0ff-Q\'\x8b\xfa\xb1/ly\x16m\x10"\x1e\xcfx\x9e\xb8\xe8\xae\x9cmV\xcb\xa1\xd5-\xdb\xa8l\xf97\x9a\xcc\xb3\xf7\xf01&\x18.\xca\xd1\x06\x97\x16\xcb9G\xc6\xb4Yn\x05M\xa7o=\xb9C\x8d\x1e\\\'\x81\xe8&z\xa9l\\\xc2p\xaf\xaa\xdc\x85W\xbd\xb3\xce\x05K\xd9\x9c\x8ce\x15\xa2\x9d\xb1\x90f\x11=\xed \x02C(Jmg\xd0J\xe9<M#\xa5\xca\x99\x0c\x99B\xc4\xc7\xde\xecN\xe6^\xd0v\xa81\xb4\x9a\x7faCl^\xc0\x91\x08\x8f\xa7T\x9d\xb1rT\xbd\xf2[_\x97\x86t\xb9=\xdf:\xd0c\x84G\x110\xe4,\xeeM\xc7\x13\xef\xaaY\\\x04<Ir\xe3L\xbd-_\xd7\xe4\xe1\xf2\xbd-XJ;\x95Q\xb1\x12\xeb\x1f_C\x8d\x08\xc3\x89\xeaJv\xe5\xb4\xb2c$\xcb+gj\xab\x1b^\x11x(\\\x01\xf6\x8f\xb6\xaaN\x8cmW"\xc87\xa8\xac\x1d\xc3\xae\xe4\x1f\xc5\xf6E\x0b\x964\t\xd0p\xdf\xa2\xb1*\xc8\xe1\xf4\xc4L\xb0Fu\x0f\xcfE\x84\xbb,\xafK\\\xcb\x81\xa4\x06,qb\x8b\xc6\xe8\x92\xfe\x80\x07\xa9\xac\x9e:_\xcf\xdc\x02\xee`d2\x95)M\xcc!\x00n\x9cP2\x0b\x8foiU\xf8\xd1\xd4DE.\x00\xe5\xe6d\xc4\xb4\xb9\xd2\x9eaU\xadRh\n\xf0l\x17\xc5\x9b^\xb1\x06\xa10\x14a\xa1\xcd\x1a\xdc\xc7\xe2\x97{d\x95\xdab\xf0\x8b\xe3\x9d\xeb\xa9\t\xbb\xd7\xde\x8ccQ\xf2\xf2\xe8\xab\xca\xeb\xbb\x8c\x08\xcc\x84\xad\xcd\xff3\x9aU\xb8\x8fI4)\x96\xfb\x15\xc7\x04{V\xb8\x93IO\xcbm\xbfu\x05\xaf\x7f\xce\xc7\xa6\xe1+\x85\x04\x05y\xc4G\x82\x80\x04\x95\x86\x82:\xaf\x92\xc7%\xdb\xd4\xc1\xb9y\xdd7\x19yjM\r\x0f\xe6\x11\n>\xdfT\x9c\x98#\xa6s\xd01gx\xf2\x00\x8f\xb1\xa1\xd4\xa7\x9a\x95\xcc/\x8b\xc4\xe2\xc5\x81L\xbe?\x1a\xf9\r\x0f\xaa+\x04\xc6\xf2\x17\xe2\xc1\xd1R\xf2\x8c\xe6\x98h\x1e\xa1\nH\x92\xad(n\xc9\xf6\xe0\x95\x89\x93\xf4`\rs\xf7\xc5,\xe1\xb9\x95\xea7\x0f\x01\x9f\xea\x90\x12\xef\x10\xbcV\x10\x8b\xc3\xda\x10\xb2-\x0f\x0f\xa1CC\xe4)"15\xc2V\x1e4K\xce\xa24\xd9\xc4\xc0\xe6\xbc\x00\x14\x0f4t\xcc!\x94\xb6\\\xc4\xd7<\xee\x11\xe3t\xf7\xef\xb8\xd3w|\x82\x17_!\xcc]\x02\x93\xce\x1f\xd0T\x06\xdfY\xd98[]\xba\x8f\xbb5\xfdX\x9e.\xa4\x9b\xae\xbc\x13\x9e\xed\xaa\x0e\xb7\x0e\x84\xdd\x00\xc3\x94\xfa\xc0\x90\x14\x9d8\x8c\x0c-Pm\xa3\xd1\x80\xf5/s\x0b|\x104\x93\x05S\x0c\xb7\xc5s\xc8\x02-\x1d\x1f\x95\xf0\xe8\x1c\x9e* \x02\xc3b-\'\xad\x91\xee\x8c\x99\xedO\x99t\xee\xe3n\xfa/R\x02N\x19qJ\x89,\x8an\xd91h\x0ey\x13r\xdbq\xd0\x07\x8d\xabd\x82\x86\x8d"\xa3y\xa0\x8c350\xc1\xc9)\xae|\x91\xca&\xb0\xbe}\x1aK\xac\xb99\xa1>\xf54\xecvo 8\xbe\xef\xd0\xb1\xa3\x87\x8f\x1dM\xec\xeb=\x020\x1f9t\xe8(\x92t\x164=\xc0\x9f\x18\xdc\x99\x108\x15\xc8\xec\xc9Q\xa5>\x1d;\xea\xe7\xb1\x11%\x84\xc0>\x07"\r\xb0\x98\x7f\xb2\x94\xf7\xd2mM\xe3\xf6\xf5\x00i\x89?F\xb5(I\xc31\xab\x98V\xe7*zf\x13\x9d\xfc\x02g<\xbb\xe7V\xbc\xa3\x02\xf1\x05\xbc\x9e\x04\xa6\x1e\x00\x13\x1aZh\x08:M\xa6\x17\x17\x0f\xee\xf9\xa2\x07\xf7\xcd\xca\x8e%\x9a[\xfb\x13\xa2\xb5\x7fT\xa7\x07\xed8H|\xd7\x01!\xba\xb4\xb6DqL\xaajv\xc0\xea\xcc\xe1\xa2]\xe2\xf6I\x1b\x98X\xec\x04 X\xc4\x92\xee\xab\xc5\xf0J\x94\x01[\xbfw\xb6\xfe\xed\x0c\xf6R\x9e\x9b\xf5\xd3\xa8\xd8\xb4nC[-\x83t\x90Ob\xf6"\xcf\xf2\xd8\x11\x1c\xe2\x02\x9a\x16I\x02\xa3\x9a\xa8\xbei\xf4\x11\x9e\x0c{zM?t\xd1\x7f\xf0\x99:\xfc\xc4\x1br\xd1\xd5ft\x827\x1fl\x97&\xc5J15<\xa8Z\xb5${G\xa6`\x90\xb5\x9cKy\xf3np\xb6\xfeJ`\xf5*\x9e\x16:\xfb\xa5$V\x1cg*m9\xf8\xcbGf\xb2EC\x0c[\x10\xc8#\xe6\r\xccv\x97\x87J\xaf0DL\xfc[\x1fE\xa9\xf9\xa0\xfa8\x1d\xa5\xderXu\xa9N\xa8\x93\xc8\x90-9x\xaameu\x86\xc31\x8cx\xff\xeb\xe43\xc3v<\x8e\xea\x87T\x0f\x1a\x8e\xfa\x17\x82\xa8i\xdfvw\xecL\x8d\xf9\xbf\x15D\xac\xe6\x10\x8a\x13\x0b\xbc\x18\ns\xa4\xc8\xf1\x89\x19\xdfAAArJ\xb4b\xc93\x0bd\x82nR!\xcf/\xa5B\xe6\xdf3c\xb5\x89\xc7\xfb\xdb\x939\xc3\xe8\x943\xa56\x9f\xb3\xde\xc6)c\xd5\x95\x9f4\x93\x98\x96\xb4\x06\xcd!\x1b{}\x85Y+\x0e\xf3\x99`\x03\x1c\x0f\xaa\xa6\x1au\xf7\xcd\x03\xd4\n\xfe>61)\x84\x15\x14\x9d\x85C\xf0\x8c\xbd\xddX\xe2\x9f\x1b\xefe\x99\x07\xb5\x00v\xdb8%\xbd\xbb\x91\xd7\xa5\xa6.K\x1b\x9d\xeao?\xc4\x91\xe3T\x17\x9a\xb42\x19V\x08\xa4\xbe\xd8\x18\x16\xfd\x83\xcad\xe0&\xde\xe7\xf7\xf4\xd9\xbc\\|\x8bS\x13V\xb1(_\xae\x93\x0e9a\xf9\xa7h\xa9\xc5\xa4Xa\x8d9\x9cc\xf1\x8eh\x03JN\xa5\xf8h\x04\xe9\xedjN0\x03,b;,\xb0qN?\x92\x02\xb5h"\xe4\xc6^\xb8\xc9\x80\x15\xd5\xd7\x1e\xa8\xeep\x06\xd8\xbf\x1f\x91\xff\xbd\xf4\xad\x81G\x1bwvba\xa4\xda\x1c\xf5`~\xde\xdb\xf5\xab\xf5\x073"\xf1\x80wwQ\x19e\x13\x06`W\xfe\x0f\xc0<\xef\xc7').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'l2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Prepare remote-only defaults for tasks not selected manually in the Notebook.
遠端 API 為尚未手動執行的 Task 建立預設參數，不影響 Notebook 既有設定。
'''
TASK_CONFIGS = globals().setdefault('TASK_CONFIGS', {})
for task_name in REMOTE_ALLOWED_TASKS:
    if task_name not in TASK_CONFIGS:
        defaults = dict(prompt='A cinematic landscape with ambient sound.',
                        width=384, height=224, seconds=10, fps=24, steps=6 if REMOTE_SERVICE_ID.startswith('06_HyperFlow_') else 20,
                        seed=42, quant=QUANT, output_name=f'{task_name}.mp4',
                        first_image='', last_image='', ref_images=[], ref_videos=[], ref_audios=[],
                        video_audio=False, ref_image_size='match')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_'):
            defaults['audio_shift'] = 6.0
        else:
            defaults['encoder_variant'] = ENCODER_VARIANT
        TASK_CONFIGS[task_name] = defaults

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            selected = globals().get('YUe_DEVICE_NAME', globals().get('YUE_DEVICE_NAME'))
            if process is None or process.poll() is not None:
                return False, f'YuE2 Vulkan{gpu_id} 常駐服務尚未啟動。', []
            if not selected or selected not in [str(part) for part in process.args]:
                return False, 'YuE2 Vulkan 啟動裝置與目前設定不一致。', []
            return True, f'YuE2 Vulkan 後端已就緒（{selected}）。', [selected]

        import torch
        if not torch.cuda.is_available():
            return False, 'PyTorch ROCm／CUDA GPU 不可用，請先執行模型初始化 Cell。', []
        selected_gpus = sorted(set(globals().get('MODEL_GPU_IDS', [0]) + [globals().get('AUX_GPU_ID', 0)]))
        available = torch.cuda.device_count()
        if any(type(i) is not int or i < 0 or i >= available for i in selected_gpus):
            return False, f'指定的 GPU {selected_gpus} 超出 PyTorch 可見範圍（{available} 張）。', []
        devices = [f'cuda:{i} {torch.cuda.get_device_name(i)}' for i in selected_gpus]
        if REMOTE_KIND == 'image_multi':
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／記憶體管理器尚未初始化。', devices
            if state.get('device') != f"cuda:{globals().get('MODEL_GPU_ID', 0)}":
                return False, 'Qwen Transformer 的 GPU 配置不一致，請重新載入。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa Pipeline 尚未完整載入。', devices
            for name, module, gpu_id in (
                ('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)),
                ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 0)),
            ):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{gpu_id}'):
                    return False, f'{name} 並未載入指定的 cuda:{gpu_id}。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any('cuda' in str(item.get('type', '')).lower() or
                       'cuda' in str(item.get('name', '')).lower() or
                       'rocm' in str(item.get('name', '')).lower()
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 未回報 GPU 後端，已拒絕 CPU-only 推論。', devices
        return True, 'GPU 後端已就緒（不表示所有權重常駐 GPU）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'fps', 'steps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    if 'fps' in numbers and not 1 <= numbers['fps'] <= 60:
        raise ValueError('fps 必須介於 1～60')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        image_count = int(fields.get('_image_count', 0))
        video_count = int(fields.get('_video_count', 0))
        audio_count = int(fields.get('_audio_count', 0))
        if not (image_count or video_count):
            raise ValueError('R2V 至少需要參考圖片或影片，不能只提供音訊。')
        if image_count > 9 or video_count > 3 or audio_count > 3 or image_count + video_count + audio_count > 12:
            raise ValueError('R2V 最多 9 張圖片、3 段影片、3 段音訊，合計最多 12 個素材。')
        count_by_type = {'Picture': image_count, 'Video': video_count,
                         'Audio': audio_count + (video_count if str(fields.get('video_audio', 'false')).lower() == 'true' else 0)}
        for media, count in count_by_type.items():
            matches = [int(number) for number in re.findall(r'<%s\s+(\d+)>' % media, str(fields['prompt']), re.I)]
            if any(number < 1 or number > count for number in matches):
                raise ValueError(f'R2V Prompt 的 <{media} N> 編號超出已上傳的參考素材數量。')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'notebook-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES, 'video_limit': 3, 'audio_limit': 3,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Host disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files, videos, audios = {}, [], [], []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name in ('images', 'videos', 'audios'):
                        collection = {'images': files, 'videos': videos, 'audios': audios}[name]
                        collection.append((part.get_filename() or 'upload', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES or len(videos) > 3 or len(audios) > 3:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and (files or videos or audios):
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (0, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if task != 'r2v' and (videos or audios):
                raise ValueError('Reference videos/audio are supported only for R2V')
            if task == 'r2v' and not (files or videos):
                raise ValueError('R2V requires an image or a video; audio alone is not supported')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files), _video_count=len(videos), _audio_count=len(audios)))
            if 'quant' in fields and fields['quant']:
                active_model = globals().get('H3_ACTIVE_MODEL', ())
                if fields['quant'] not in active_model:
                    self.send_json(409, {'error': '指定的量化尚未載入；請先在 Notebook 下載並預檢該 GGUF。'})
                    return
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            if str(fields.get('video_audio', 'false')).lower() not in ('true', 'false'):
                raise ValueError('video_audio must be true or false')
            for filename, data in videos:
                if not data or len(data) > 80 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp4', '.mov', '.webm'):
                    raise ValueError('Reference video must be MP4/MOV/WebM and at most 80 MiB')
            for filename, data in audios:
                if not data or len(data) > 30 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp3', '.wav', '.flac', '.ogg', '.m4a'):
                    raise ValueError('Reference audio must be an audio file at most 30 MiB')
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names, video_names, audio_names = [], [], []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            for kind, uploaded, names in (('video', videos, video_names), ('audio', audios, audio_names)):
                for index, (filename, data) in enumerate(uploaded, 1):
                    target = staging / f'{kind}_{index:03d}{Path(filename).suffix.lower()}'
                    target.write_bytes(data)
                    names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    fields['_ref_videos'] = [str(folder / name) for name in video_names]
                    fields['_ref_audios'] = [str(folder / name) for name in audio_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'FPS', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                'QUANT', 'ENCODER_VARIANT']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'fps': int, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'fps' in inputs and not 1 <= inputs['fps'] <= 60:
                raise ValueError('FPS must be between 1 and 60')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'fps', 'steps', 'seed',
                                  'audio_shift', 'ref_image_size', 'quant'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    updated['quant'] = inputs.get('quant') or namespace.get('QUANT', updated.get('quant'))
                    if task == 'r2v':
                        updated['video_audio'] = str(inputs.get('video_audio', 'false')).lower() == 'true'
                        updated['ref_videos'] = inputs.get('_ref_videos', [])
                        updated['ref_audios'] = inputs.get('_ref_audios', [])
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'l2v':
                        updated['last_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = REMOTE_ROOT / job_id
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 14 節完成：選用 FIFO／ngrok API 已啟動。')


15. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_MiniMax_H3_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 15 節完成：API Schema 和 client 說明已顯示。')
